In [ ]:
### Geolibraries
import geopandas as gpd
import osmnx as ox
import contextily as ctx; import basemaps


# General tools
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pyarrow.parquet as pq

from h3 import h3
from shapely.geometry import Polygon
import mapclassify as mc


In [ ]:
users = pd.read_parquet("./data/user_pois_pt_1.parquet")

user_home_hex = (
    users[users["is_home"] == 1]
    .drop_duplicates(subset="user_id")
)

In [ ]:
user_work_hex = (
    users[users["is_work"] == 1]
    .drop_duplicates(subset="user_id")
)


In [ ]:
### Violin plot

In [ ]:
df_jobs = pd.read_parquet("scratch/pt_typ_cat.parquet") # go to by user per mode and city and tun to get this

In [ ]:
df_jobs

In [ ]:
df_jobs_cars = pd.read_parquet("scratch/car_typ_cat.parquet")

In [ ]:
df_jobs_bike = pd.read_parquet("scratch/bike_typ_cat.parquet")

In [ ]:
df_jobs_bike

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# -------------------------------------------------
# FILTER ONLY JOBS
# -------------------------------------------------

jobs_pt = (
    df_jobs[df_jobs["poi_type"] == "jobs"]
    .copy()
)

jobs_car = (
    df_jobs_cars[df_jobs_cars["poi_type"] == "jobs"]
    .copy()
)

jobs_bike = (
    df_jobs_bike[df_jobs_bike["poi_type"] == "jobs"]
    .copy()
)

# -------------------------------------------------
# CLEAN VALUES
#
# remove:
# - NaN
# - zeros
# - unrealistically small trips


pt_vals = (
    jobs_pt["typical_trip_co2"]
    .dropna()
)

pt_vals = pt_vals[pt_vals >= 15] / 1000  # kg

car_vals = (
    jobs_car["typical_trip_co2"]
    .dropna()
)

car_vals = car_vals[car_vals >= 35] / 1000  # kg

bike_vals = (
    jobs_bike["typical_trip_co2"]
    .dropna()
)

bike_vals = bike_vals[bike_vals >= 10] / 1000  # kg

# -------------------------------------------------
# VISUALIZATION FILTER
#
# Remove extreme car tail ONLY for plotting
# -------------------------------------------------

car_vis = car_vals[car_vals <= 7]

pt_vis = pt_vals.copy()
bike_vis = bike_vals.copy()

# -------------------------------------------------
# COMBINED DATAFRAME FOR PLOTTING
# -------------------------------------------------

plot_df = pd.concat([

    pd.DataFrame({
        "Mode": "Bike",
        "CO2": bike_vis
    }),

    pd.DataFrame({
        "Mode": "Public Transport",
        "CO2": pt_vis
    }),

    pd.DataFrame({
        "Mode": "Car",
        "CO2": car_vis
    })

], ignore_index=True)

# -------------------------------------------------
# STYLE
# -------------------------------------------------

sns.set_style("white")

colors = {
    "Bike": "#2b8cbe",
    "Public Transport": "#7bccc4",
    "Car": "#de2d26"
}

mode_order = [
    "Bike",
    "Public Transport",
    "Car"
]

# -------------------------------------------------
# FIGURE
# -------------------------------------------------

fig, ax = plt.subplots(figsize=(8.5, 4.5))

# -------------------------------------------------
# VIOLINS
# -------------------------------------------------

sns.violinplot(
    data=plot_df,
    y="Mode",
    x="CO2",
    order=mode_order,
    orient="h",
    palette=colors,
    inner=None,
    linewidth=0,
    cut=0,
    bw_adjust=0.9,
    saturation=1,
    width=0.9,
    ax=ax
)

# -------------------------------------------------
# SOFT TRANSPARENCY
# -------------------------------------------------

for collection in ax.collections:
    collection.set_alpha(0.38)

# -------------------------------------------------
# MEDIANS
#
# IMPORTANT:
# computed from ORIGINAL distributions
# not visualization-filtered values
# -------------------------------------------------

medians = {
    "Bike": bike_vals.median(),
    "Public Transport": pt_vals.median(),
    "Car": car_vals.median()
}

for i, mode in enumerate(mode_order):

    median_val = medians[mode]

    # keep point inside visible axis
    visible_x = min(median_val, 7)

    # median point
    ax.scatter(
        visible_x,
        i,
        s=95,
        color="black",
        zorder=10
    )

    # median label
    ax.text(
        visible_x + 0.10,
        i,
        f"{median_val:.2f} kg",
        va="center",
        ha="left",
        fontsize=9,
        fontweight="bold",
        color="black"
    )

# -------------------------------------------------
# AXES
# -------------------------------------------------

ax.set_xlim(0, 7)

ax.set_xticks(
    np.arange(0, 8, 1)
)

ax.set_xlabel(
    "Commuting CO₂ expenditure per trip (kg)"
)

ax.set_ylabel("")

ax.set_title(
    "Distribution of commuting CO₂ expenditure by transport mode",
    pad=14
)

# -------------------------------------------------
# SUBTLE GRID
# -------------------------------------------------

ax.xaxis.grid(
    True,
    linestyle="--",
    linewidth=0.5,
    alpha=0.22
)

ax.yaxis.grid(False)

# -------------------------------------------------
# CLEAN STYLE
# -------------------------------------------------

sns.despine(
    left=False,
    bottom=False
)

plt.tight_layout()

# -------------------------------------------------
# SAVE
# -------------------------------------------------

plt.savefig(
    "commuting_co2_violin_clean_jobs.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# -------------------------------------------------
# OPTIONAL REPORT
# -------------------------------------------------

print("Observations used in visualization")
print("----------------------------------")
print(f"Bike: {len(bike_vis):,}")
print(f"PT:   {len(pt_vis):,}")
print(f"Car:  {len(car_vis):,}")

print("\nRemoved from car visualization (>7 kg):")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------
# LOAD DATA (HELSINKI)
# -------------------------------------------------
df_jobs_bike = df_jobs_bike.copy()
df_jobs_car = df_jobs_cars.copy()
df_jobs_pt = df_jobs.copy()

# -------------------------------------------------
# PURPOSE WEIGHTS
# -------------------------------------------------
purpose_weights = {
    "jobs": 4,
    "Recreational, Outdoors": 2,
    "Social, Cultural": 2,
    "Shopping, Errands": 1
}

required_purposes = list(purpose_weights.keys())

# -------------------------------------------------
# BUILD PER-USER DECENT MOBILITY INDEX
# -------------------------------------------------
def build_user_index(df):

    df = df.copy()
    df["co2_kg"] = df["typical_trip_co2"] / 1000

    # user × purpose matrix
    pivot = df.pivot_table(
        index="user_id",
        columns="poi_type",
        values="co2_kg",
        aggfunc="first"
    )

    # STRICT: only complete users
    pivot = pivot.dropna(subset=required_purposes)

    # apply weights
    for col in required_purposes:
        pivot[col] = pivot[col] * purpose_weights[col]

    total = pivot.sum(axis=1)
    commuting = pivot["jobs"]
    non_work = total - commuting

    return commuting, non_work

# -------------------------------------------------
# COMPUTE PER MODE
# -------------------------------------------------
bike_comm, bike_non = build_user_index(df_jobs_bike)
pt_comm, pt_non     = build_user_index(df_jobs_pt)
car_comm, car_non   = build_user_index(df_jobs_car)

# -------------------------------------------------
# MEDIANS (CONSISTENT ACROSS MODES)
# -------------------------------------------------
order = ["Bike", "Public Transport", "Car"]

data = pd.DataFrame({
    "Mode": order,
    "Commuting": [
        bike_comm.median(),
        pt_comm.median(),
        car_comm.median()
    ],
    "NonWork": [
        bike_non.median(),
        pt_non.median(),
        car_non.median()
    ]
})

# -------------------------------------------------
# SHARES
# -------------------------------------------------
data["Total"] = data["Commuting"] + data["NonWork"]
data["Commuting_pct"] = data["Commuting"] / data["Total"] * 100
data["NonWork_pct"] = data["NonWork"] / data["Total"] * 100

# -------------------------------------------------
# PLOT
# -------------------------------------------------
colors = {
    "commuting": "#1f2a44",
    "non_work": "#4c78a8"
}

fig, ax = plt.subplots(figsize=(11, 4.4))

y = np.arange(len(order))
bar_h = 0.38

# stacked bars
ax.barh(y, data["Commuting_pct"], height=bar_h, color=colors["commuting"])
ax.barh(
    y,
    data["NonWork_pct"],
    left=data["Commuting_pct"],
    height=bar_h,
    color=colors["non_work"]
)

# -------------------------------------------------
# LABELS
# -------------------------------------------------
for i in range(len(order)):

    c = data["Commuting_pct"].iloc[i]
    n = data["NonWork_pct"].iloc[i]
    t = data["Total"].iloc[i]

    ax.text(
        c / 2, i,
        f"{c:.0f}%",
        ha="center",
        va="center",
        color="white",
        fontweight="bold"
    )

    ax.text(
        c + n / 2, i,
        f"{n:.0f}%",
        ha="center",
        va="center",
        color="white",
        fontweight="bold"
    )

    ax.text(
        102, i,
        f"Total: {t:.2f} kg CO₂",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

# -------------------------------------------------
# FORMATTING
# -------------------------------------------------
ax.set_yticks(y)
ax.set_yticklabels(order)

ax.set_xlim(0, 110)
ax.set_xlabel("Share of CO₂ footprint per trip purpose (%)")

ax.set_title(
    "Decent mobility CO₂ composition (Helsinki)\nPer-user median-based index",
    pad=14
)

ax.xaxis.grid(True, linestyle="--", alpha=0.25)
ax.yaxis.grid(False)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

# -------------------------------------------------
# Legend below the plot
# -------------------------------------------------
ax.legend(
    ["Commuting (jobs)", "Non-work trips"],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.28),
    ncol=2,
    frameon=False
)

# reserve space for legend
plt.subplots_adjust(bottom=0.25, top=0.88)

plt.tight_layout()

# -------------------------------------------------
# SAVE
# -------------------------------------------------
plt.savefig(
    "helsinki_decent_mobility_FIXED.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
home_counts = (
    user_home_hex.groupby("home_gid9")
      .size()
      .reset_index(name="n_users")
)

work_counts = (
    user_work_hex.groupby("work_gid9")
      .size()
      .reset_index(name="n_users")
)

In [ ]:
def h3_to_polygon(h):
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)
    return Polygon(boundary)

In [ ]:
home_counts["geometry"] = home_counts["home_gid9"].apply(h3_to_polygon)

home_gdf = gpd.GeoDataFrame(home_counts, geometry="geometry", crs="EPSG:4326")

In [ ]:
work_counts["geometry"] = work_counts["work_gid9"].apply(h3_to_polygon)

work_gdf = gpd.GeoDataFrame(work_counts, geometry="geometry", crs="EPSG:4326")

In [ ]:
home_gdf = home_gdf.to_crs(epsg=3067)
work_gdf = work_gdf.to_crs(epsg=3067)

In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import numpy as np
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter

# -------------------------------------------------
# 1. PROJECT CRS
# -------------------------------------------------
home_plot = home_gdf.to_crs(epsg=3857).copy()
work_plot = work_gdf.to_crs(epsg=3857).copy()

# -------------------------------------------------
# 2. HARD SCALE CAP (ONLY CHANGE)
# -------------------------------------------------
CAP = 140

home_plot["clip"] = home_plot["n_users"].clip(upper=CAP)
work_plot["clip"] = work_plot["n_users"].clip(upper=CAP)

# -------------------------------------------------
# 3. SMOOTHING
# -------------------------------------------------
home_plot["smooth"] = gaussian_filter(home_plot["clip"].values, sigma=1)
work_plot["smooth"] = gaussian_filter(work_plot["clip"].values, sigma=1)

# -------------------------------------------------
# 4. Normalisation (fixed 0–150 scale)
# -------------------------------------------------
norm = Normalize(vmin=0, vmax=CAP)

# -------------------------------------------------
# 5. PLOT
# -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# ---------------- HOME ----------------
home_plot.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[0],
    norm=norm
)

home_plot.boundary.plot(
    ax=axes[0],
    color="darkgray",
    linewidth=0.3,
    alpha=0.9
)

ctx.add_basemap(
    axes[0],
    source=basemaps.DARK_NOLABELS
)

axes[0].set_title("Home locations (Helsinki)", fontsize=11)

# ---------------- WORK ----------------
work_plot.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[1],
    norm=norm
)

work_plot.boundary.plot(
    ax=axes[1],
    color="darkgray",
    linewidth=0.3,
    alpha=0.9
)

ctx.add_basemap(
    axes[1],
    source=basemaps.DARK_NOLABELS
)

axes[1].set_title("Work locations (Helsinki)", fontsize=11)

# ---------------- CLEAN ----------------
for ax in axes:
    ax.axis("off")

plt.tight_layout()

plt.savefig(
    "./output/home_work_helsinki_users_150.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
home_gdf.to_parquet("./data/hex_filter_hsk.parquet")

In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import numpy as np
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter

# -------------------------------------------------
# 1. PROJECT CRS
# -------------------------------------------------
home_plot = home_gdf.to_crs(epsg=3857).copy()
work_plot = work_gdf.to_crs(epsg=3857).copy()

# -------------------------------------------------
# 2. HARD CAP AT 150 (NEW)
# -------------------------------------------------
CAP = 140

home_plot["clip"] = home_plot["n_users"].clip(upper=CAP)
work_plot["clip"] = work_plot["n_users"].clip(upper=CAP)

# -------------------------------------------------
# 3. SMOOTHING
# -------------------------------------------------
home_plot["smooth"] = gaussian_filter(home_plot["clip"].values, sigma=1)
work_plot["smooth"] = gaussian_filter(work_plot["clip"].values, sigma=1)

# -------------------------------------------------
# 4. Normalisation (fixed 0–150 scale)
# -------------------------------------------------
norm = Normalize(vmin=0, vmax=CAP)

# -------------------------------------------------
# 5. PLOT
# -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# ---------------- HOME ----------------
home_plot.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[0],
    norm=norm
)

home_plot.boundary.plot(
    ax=axes[0],
    color="darkgray",
    linewidth=0.3,
    alpha=0.9
)

ctx.add_basemap(
    axes[0],
    source=basemaps.POSITRON_NOLABELS
)

axes[0].set_title("Home locations (Helsinki)", fontsize=11)

# ---------------- WORK ----------------
work_plot.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[1],
    norm=norm
)

work_plot.boundary.plot(
    ax=axes[1],
    color="darkgray",
    linewidth=0.3,
    alpha=0.9
)

ctx.add_basemap(
    axes[1],
    source=basemaps.POSITRON_NOLABELS
)

axes[1].set_title("Work locations (Helsinki)", fontsize=11)

# -------------------------------------------------
# CLEAN AXES
# -------------------------------------------------
for ax in axes:
    ax.axis("off")

# -------------------------------------------------
# SPACE FOR COLORBAR (RIGHT SIDE)
# -------------------------------------------------
fig.subplots_adjust(right=0.88)

# -------------------------------------------------
# COLORBAR (VERTICAL OUTSIDE)
# -------------------------------------------------
sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=norm)
sm._A = []

cbar_ax = fig.add_axes([0.90, 0.2, 0.02, 0.6])

cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("Frequency", fontsize=10)

# -------------------------------------------------
# SAVE
# -------------------------------------------------
plt.savefig(
    "./output/home_work_helsinki_users_cap150.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter

# -------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------
users_turku = pd.read_parquet("./data/user_pois_pt_1_turku.parquet")

# -------------------------------------------------
# 2. FILTER HOME / WORK
# -------------------------------------------------
user_home_hex_turku = (
    users_turku[users_turku["is_home"] == 1]
    .drop_duplicates(subset="user_id")
)

user_work_hex_turku = (
    users_turku[users_turku["is_work"] == 1]
    .drop_duplicates(subset="user_id")
)

# -------------------------------------------------
# 3. AGGREGATION
# -------------------------------------------------
home_counts_turku = (
    user_home_hex_turku.groupby("home_gid9")
    .size()
    .reset_index(name="n_users")
)

work_counts_turku = (
    user_work_hex_turku.groupby("work_gid9")
    .size()
    .reset_index(name="n_users")
)

# -------------------------------------------------
# 4. H3 → GEOMETRY
# -------------------------------------------------
home_counts_turku["geometry"] = home_counts_turku["home_gid9"].apply(h3_to_polygon)
work_counts_turku["geometry"] = work_counts_turku["work_gid9"].apply(h3_to_polygon)

home_gdf_turku = gpd.GeoDataFrame(
    home_counts_turku,
    geometry="geometry",
    crs="EPSG:4326"
)

work_gdf_turku = gpd.GeoDataFrame(
    work_counts_turku,
    geometry="geometry",
    crs="EPSG:4326"
)

# -------------------------------------------------
# 5. PROJECT CRS
# -------------------------------------------------
home_plot_turku = home_gdf_turku.to_crs(epsg=3857).copy()
work_plot_turku = work_gdf_turku.to_crs(epsg=3857).copy()

# -------------------------------------------------
# 6. FILTER SELECTION POLYGON
# -------------------------------------------------
filter_poly = gpd.read_file("./data/filter_hex_turku.geojson").to_crs(epsg=3857)
selection_geom = filter_poly.unary_union

home_plot_turku = home_plot_turku[
    home_plot_turku.geometry.centroid.within(selection_geom)
].copy()

work_plot_turku = work_plot_turku[
    work_plot_turku.geometry.centroid.within(selection_geom)
].copy()

# -------------------------------------------------
# 7. HARD CAP AT 150 (IMPORTANT CHANGE)
# -------------------------------------------------
CAP = 130

home_plot_turku["clip"] = home_plot_turku["n_users"].clip(upper=CAP)
work_plot_turku["clip"] = work_plot_turku["n_users"].clip(upper=CAP)

# -------------------------------------------------
# 8. SMOOTHING
# -------------------------------------------------
home_plot_turku["smooth"] = gaussian_filter(home_plot_turku["clip"].values, sigma=1)
work_plot_turku["smooth"] = gaussian_filter(work_plot_turku["clip"].values, sigma=1)

# -------------------------------------------------
# 9. Normalisation (fixed 0–150 scale)
# -------------------------------------------------
norm_turku = Normalize(vmin=0, vmax=CAP)

# -------------------------------------------------
# 10. PLOT
# -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# HOME
home_plot_turku.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[0],
    norm=norm_turku
)

home_plot_turku.boundary.plot(
    ax=axes[0],
    color="darkgray",
    linewidth=0.4,
    alpha=0.9
)

ctx.add_basemap(
    axes[0],
    source=basemaps.DARK_NOLABELS
)

axes[0].set_title("Home locations (Turku)", fontsize=11)

# WORK
work_plot_turku.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[1],
    norm=norm_turku
)

work_plot_turku.boundary.plot(
    ax=axes[1],
    color="darkgray",
    linewidth=0.4,
    alpha=0.9
)

ctx.add_basemap(
    axes[1],
    source=basemaps.DARK_NOLABELS
)

axes[1].set_title("Work locations (Turku)", fontsize=11)

# -------------------------------------------------
# CLEAN
# -------------------------------------------------
for ax in axes:
    ax.axis("off")

plt.tight_layout()

# -------------------------------------------------
# SAVE
# -------------------------------------------------
plt.savefig(
    "./output/home_work_turku_users_cap150.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter

# -------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------
users_turku = pd.read_parquet("./data/user_pois_pt_1_turku.parquet")

# -------------------------------------------------
# 2. FILTER HOME / WORK
# -------------------------------------------------
user_home_hex_turku = (
    users_turku[users_turku["is_home"] == 1]
    .drop_duplicates(subset="user_id")
)

user_work_hex_turku = (
    users_turku[users_turku["is_work"] == 1]
    .drop_duplicates(subset="user_id")
)

# -------------------------------------------------
# 3. AGGREGATION
# -------------------------------------------------
home_counts_turku = (
    user_home_hex_turku.groupby("home_gid9")
    .size()
    .reset_index(name="n_users")
)

work_counts_turku = (
    user_work_hex_turku.groupby("work_gid9")
    .size()
    .reset_index(name="n_users")
)

# -------------------------------------------------
# 4. H3 → GEOMETRY
# -------------------------------------------------
home_counts_turku["geometry"] = home_counts_turku["home_gid9"].apply(h3_to_polygon)
work_counts_turku["geometry"] = work_counts_turku["work_gid9"].apply(h3_to_polygon)

home_gdf_turku = gpd.GeoDataFrame(
    home_counts_turku,
    geometry="geometry",
    crs="EPSG:4326"
)

work_gdf_turku = gpd.GeoDataFrame(
    work_counts_turku,
    geometry="geometry",
    crs="EPSG:4326"
)

# -------------------------------------------------
# 5. PROJECT CRS
# -------------------------------------------------
home_plot_turku = home_gdf_turku.to_crs(epsg=3857).copy()
work_plot_turku = work_gdf_turku.to_crs(epsg=3857).copy()

# -------------------------------------------------
# 6. FILTER SELECTION POLYGON
# -------------------------------------------------
filter_poly = gpd.read_file("./data/filter_hex_turku.geojson").to_crs(epsg=3857)
selection_geom = filter_poly.unary_union

home_plot_turku = home_plot_turku[
    home_plot_turku.geometry.centroid.within(selection_geom)
].copy()

work_plot_turku = work_plot_turku[
    work_plot_turku.geometry.centroid.within(selection_geom)
].copy()

# -------------------------------------------------
# 7. CLIP
# -------------------------------------------------
all_vals_turku = np.concatenate([
    home_plot_turku["n_users"].values,
    work_plot_turku["n_users"].values
])

vmax_turku = np.percentile(all_vals_turku, 99.5)
vmin_turku = 0

home_plot_turku["clip"] = home_plot_turku["n_users"].clip(upper=vmax_turku)
work_plot_turku["clip"] = work_plot_turku["n_users"].clip(upper=vmax_turku)

# -------------------------------------------------
# 8. SMOOTHING
# -------------------------------------------------
home_plot_turku["smooth"] = gaussian_filter(home_plot_turku["clip"].values, sigma=1)
work_plot_turku["smooth"] = gaussian_filter(work_plot_turku["clip"].values, sigma=1)

# -------------------------------------------------
# 9. NORMALIZATION
# -------------------------------------------------
norm_turku = Normalize(vmin=vmin_turku, vmax=vmax_turku)

# -------------------------------------------------
# 10. PLOT
# -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# ---------------- HOME ----------------
home_plot_turku.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[0],
    norm=norm_turku
)

home_plot_turku.boundary.plot(
    ax=axes[0],
    color="darkgray",
    linewidth=0.4,
    alpha=0.9
)

ctx.add_basemap(
    axes[0],
    source=basemaps.POSITRON_NOLABELS
)

axes[0].set_title("Home locations (Turku)", fontsize=11)

# ---------------- WORK ----------------
work_plot_turku.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[1],
    norm=norm_turku
)

work_plot_turku.boundary.plot(
    ax=axes[1],
    color="darkgray",
    linewidth=0.4,
    alpha=0.9
)

ctx.add_basemap(
    axes[1],
    source=basemaps.POSITRON_NOLABELS
)

axes[1].set_title("Work locations (Turku)", fontsize=11)

# -------------------------------------------------
# CLEAN AXES
# -------------------------------------------------
for ax in axes:
    ax.axis("off")

# -------------------------------------------------
# SPACE FOR COLORBAR
# -------------------------------------------------
fig.subplots_adjust(right=0.88)

# -------------------------------------------------
# VERTICAL COLORBAR (OUTSIDE)
# -------------------------------------------------
sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=norm_turku)
sm._A = []

cbar_ax = fig.add_axes([0.90, 0.2, 0.02, 0.6])  # left, bottom, width, height

cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("Frequency", fontsize=10)

# -------------------------------------------------
# SAVE FIGURE
# -------------------------------------------------
plt.savefig(
    "./output/home_work_turku_users.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import folium
from folium.plugins import Draw

gdf = home_plot_turku

m = folium.Map(
    location=[61.45, 23.85],
    zoom_start=11,
    tiles="cartodbpositron"
)

# ADD HEXAGONS
folium.GeoJson(
    gdf,
    name="hexagons"
).add_to(m)

# DRAW TOOL
Draw(export=True).add_to(m)

m

In [ ]:
filter_poly= gpd.read_file("./data/filter_hex_turku.geojson")

gdf = home_plot_turku.to_crs(4326)
filter_poly = filter_poly.to_crs(4326)

selection_geom = filter_poly.unary_union

selected_hex = gdf[gdf.centroid.within(selection_geom)]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# -------------------------------------------------
# CONFIG
# -------------------------------------------------

CITY = "turku"

file_map = {
    "pt": f"scratch/pt_typ_cat_{CITY}.parquet",
    "car": f"scratch/car_typ_cat_{CITY}.parquet",
    "bike": f"scratch/bike_typ_cat_{CITY}.parquet"
}

# -------------------------------------------------
# LOAD DATA (suffix-driven)
# -------------------------------------------------

dfs = {
    mode: pd.read_parquet(path)
    for mode, path in file_map.items()
}

# -------------------------------------------------
# FILTER ONLY JOBS
# -------------------------------------------------

jobs = {
    mode: df[df["poi_type"] == "jobs"].copy()
    for mode, df in dfs.items()
}

# -------------------------------------------------
# CLEANING RULES (centralized)
# -------------------------------------------------

def clean_series(df, col, min_val, scale=1000):
    s = df[col].dropna()
    s = s[s >= min_val] / scale
    return s

pt_vals = clean_series(jobs["pt"], "typical_trip_co2", 15)
car_vals = clean_series(jobs["car"], "typical_trip_co2", 35)
bike_vals = clean_series(jobs["bike"], "typical_trip_co2", 10)

# -------------------------------------------------
# VISUAL FILTERING (plot only)
# -------------------------------------------------

car_vis = car_vals[car_vals <= 7]
pt_vis = pt_vals.copy()
bike_vis = bike_vals.copy()

# -------------------------------------------------
# PLOT DATAFRAME
# -------------------------------------------------

plot_df = pd.concat([
    pd.DataFrame({"Mode": "Bike", "CO2": bike_vis}),
    pd.DataFrame({"Mode": "Public Transport", "CO2": pt_vis}),
    pd.DataFrame({"Mode": "Car", "CO2": car_vis})
], ignore_index=True)

# -------------------------------------------------
# STYLE
# -------------------------------------------------

sns.set_style("white")

colors = {
    "Bike": "#2b8cbe",
    "Public Transport": "#7bccc4",
    "Car": "#de2d26"
}

mode_order = ["Bike", "Public Transport", "Car"]

# -------------------------------------------------
# FIGURE
# -------------------------------------------------

fig, ax = plt.subplots(figsize=(8.5, 4.5))

sns.violinplot(
    data=plot_df,
    y="Mode",
    x="CO2",
    order=mode_order,
    orient="h",
    palette=colors,
    inner=None,
    linewidth=0,
    cut=0,
    bw_adjust=0.9,
    saturation=1,
    width=0.9,
    ax=ax
)

# soften violins
for c in ax.collections:
    c.set_alpha(0.38)

# -------------------------------------------------
# MEDIANS (true values)
# -------------------------------------------------

medians = {
    "Bike": bike_vals.median(),
    "Public Transport": pt_vals.median(),
    "Car": car_vals.median()
}

for i, mode in enumerate(mode_order):
    m = medians[mode]
    x = min(m, 7)

    ax.scatter(x, i, s=95, color="black", zorder=10)

    ax.text(
        x + 0.10,
        i,
        f"{m:.2f} kg",
        va="center",
        ha="left",
        fontsize=9,
        fontweight="bold"
    )

# -------------------------------------------------
# AXES
# -------------------------------------------------

ax.set_xlim(0, 7)
ax.set_xticks(np.arange(0, 8, 1))
ax.set_xlabel("Commuting CO₂ per trip (kg)")
ax.set_ylabel("")
ax.set_title(f"{CITY.title()}: CO₂ distribution by transport mode", pad=14)

ax.xaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.22)
ax.yaxis.grid(False)

sns.despine(left=False, bottom=False)

plt.tight_layout()

# -------------------------------------------------
# SAVE
# -------------------------------------------------

plt.savefig(
    f"{CITY}_commuting_co2_violin_jobs.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# -------------------------------------------------
# SUMMARY
# -------------------------------------------------

print("Observations used in visualization")
print("----------------------------------")
for mode in ["bike", "pt", "car"]:
    print(f"{mode.upper()}: {len(jobs[mode][jobs[mode]['typical_trip_co2'].notna()]):,}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------
# LOAD DATA (TURKU)
# -------------------------------------------------
df_jobs_bike_turku = pd.read_parquet("scratch/bike_typ_cat_turku.parquet")
df_jobs_car_turku  = pd.read_parquet("scratch/car_typ_cat_turku.parquet")
df_jobs_turku      = pd.read_parquet("scratch/pt_typ_cat_turku.parquet")

# -------------------------------------------------
# PURPOSE WEIGHTS
# -------------------------------------------------
purpose_weights = {
    "jobs": 4,
    "Recreational, Outdoors": 2,
    "Social, Cultural": 2,
    "Shopping, Errands": 1
}

# -------------------------------------------------
# MEDIAN-BASED COMPONENTS
# -------------------------------------------------
def compute_components(df, mode):

    rows = []

    for purpose, weight in purpose_weights.items():

        subset = df[df["poi_type"] == purpose]["typical_trip_co2"].dropna()
        subset = subset / 1000  # g → kg

        value = subset.median()

        rows.append({
            "Mode": mode,
            "Purpose": purpose,
            "Value": value * weight
        })

    return pd.DataFrame(rows)

# -------------------------------------------------
# BUILD DATA
# -------------------------------------------------
bike = compute_components(df_jobs_bike_turku, "Bike")
pt   = compute_components(df_jobs_turku, "Public Transport")
car  = compute_components(df_jobs_car_turku, "Car")

df = pd.concat([bike, pt, car])

# -------------------------------------------------
# PIVOT
# -------------------------------------------------
pivot = df.pivot_table(
    index="Mode",
    columns="Purpose",
    values="Value",
    aggfunc="sum"
).fillna(0)

order = ["Bike", "Public Transport", "Car"]
pivot = pivot.loc[order]

# -------------------------------------------------
# SPLIT
# -------------------------------------------------
commuting = pivot["jobs"]
non_work = pivot.drop(columns=["jobs"]).sum(axis=1)

total = commuting + non_work

commuting_pct = (commuting / total) * 100
non_work_pct = (non_work / total) * 100

# -------------------------------------------------
# COLORS
# -------------------------------------------------
colors = {
    "commuting": "#1f2a44",
    "non_work": "#4c78a8"
}

# -------------------------------------------------
# FIGURE
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 4.4))

y = np.arange(len(order))
bar_h = 0.38

# bars
ax.barh(y, commuting_pct.values, height=bar_h, color=colors["commuting"])
ax.barh(
    y,
    non_work_pct.values,
    left=commuting_pct.values,
    height=bar_h,
    color=colors["non_work"]
)

# -------------------------------------------------
# LABELS
# -------------------------------------------------
for i in range(len(order)):

    c = commuting_pct.iloc[i]
    n = non_work_pct.iloc[i]
    t = total.iloc[i]

    ax.text(c / 2, i, f"{c:.0f}%",
            ha="center", va="center",
            color="white", fontweight="bold")

    ax.text(c + n / 2, i, f"{n:.0f}%",
            ha="center", va="center",
            color="white", fontweight="bold")

    ax.text(102, i, f"Total: {t:.2f} kg CO₂",
            va="center", fontsize=10, fontweight="bold")

# -------------------------------------------------
# FORMATTING
# -------------------------------------------------
ax.set_yticks(y)
ax.set_yticklabels(order)

ax.set_xlim(0, 110)
ax.set_xlabel("Share of CO₂ footprint per trip purpose (%)")

ax.set_title(
    "Decent mobility CO₂ composition (Turku)\nPer-user median-based index",
    pad=14
)

ax.xaxis.grid(True, linestyle="--", alpha=0.25)
ax.yaxis.grid(False)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

# -------------------------------------------------
# Legend below the plot
# -------------------------------------------------
ax.legend(
    ["Commuting (jobs)", "Non-work trips"],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.28),
    ncol=2,
    frameon=False
)

plt.subplots_adjust(bottom=0.25, top=0.88)
plt.tight_layout()

# -------------------------------------------------
# SAVE
# -------------------------------------------------
plt.savefig(
    "decent_mobility_composition_turku_FIXED.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
df_jobs_car_turku

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter

# -------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------
users_tampere = pd.read_parquet("./data/user_pois_pt_1_tampere.parquet")

# -------------------------------------------------
# 2. FILTER HOME / WORK
# -------------------------------------------------
user_home_hex_tampere = (
    users_tampere[users_tampere["is_home"] == 1]
    .drop_duplicates(subset="user_id")
)

user_work_hex_tampere = (
    users_tampere[users_tampere["is_work"] == 1]
    .drop_duplicates(subset="user_id")
)

# -------------------------------------------------
# 3. AGGREGATION
# -------------------------------------------------
home_counts_tampere = (
    user_home_hex_tampere.groupby("home_gid9")
    .size()
    .reset_index(name="n_users")
)

work_counts_tampere = (
    user_work_hex_tampere.groupby("work_gid9")
    .size()
    .reset_index(name="n_users")
)

# -------------------------------------------------
# 4. H3 → GEOMETRY
# -------------------------------------------------
home_counts_tampere["geometry"] = home_counts_tampere["home_gid9"].apply(h3_to_polygon)
work_counts_tampere["geometry"] = work_counts_tampere["work_gid9"].apply(h3_to_polygon)

home_gdf_tampere = gpd.GeoDataFrame(
    home_counts_tampere,
    geometry="geometry",
    crs="EPSG:4326"
)

work_gdf_tampere = gpd.GeoDataFrame(
    work_counts_tampere,
    geometry="geometry",
    crs="EPSG:4326"
)

# -------------------------------------------------
# 5. PROJECT CRS
# -------------------------------------------------
home_plot_tampere = home_gdf_tampere.to_crs(epsg=3857).copy()
work_plot_tampere = work_gdf_tampere.to_crs(epsg=3857).copy()

# -------------------------------------------------
# 6. 99th PERCENTILE CAP
# -------------------------------------------------
all_vals_tampere = np.concatenate([
    home_plot_tampere["n_users"].values,
    work_plot_tampere["n_users"].values
])

vmax_tampere = np.percentile(all_vals_tampere, 99)
vmin_tampere = 0

home_plot_tampere["clip"] = home_plot_tampere["n_users"].clip(upper=vmax_tampere)
work_plot_tampere["clip"] = work_plot_tampere["n_users"].clip(upper=vmax_tampere)

# -------------------------------------------------
# 7. GAUSSIAN SMOOTHING
# -------------------------------------------------
home_plot_tampere["smooth"] = gaussian_filter(
    home_plot_tampere["clip"].values,
    sigma=1
)

work_plot_tampere["smooth"] = gaussian_filter(
    work_plot_tampere["clip"].values,
    sigma=1
)

# -------------------------------------------------
# 8. PLOT
# -------------------------------------------------
norm_tampere = Normalize(vmin=vmin_tampere, vmax=vmax_tampere)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
home_plot_tampere.to_parquet("./data/home_plot_tampere_no_filter.parquet")
# HOME
home_plot_tampere.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.85,
    ax=axes[0],
    norm=norm_tampere
)

ctx.add_basemap(
    axes[0],
    source=basemaps.DARK_NOLABELS
)

axes[0].set_title("Home locations (Tampere)", fontsize=11)

# WORK
work_plot_tampere.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.85,
    ax=axes[1],
    norm=norm_tampere
)

ctx.add_basemap(
    axes[1],
    source=basemaps.DARK_NOLABELS
)

axes[1].set_title("Work locations (Tampere)", fontsize=11)

# CLEAN
for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter

# -------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------
users_tampere = pd.read_parquet("./data/user_pois_pt_1_tampere.parquet")

# -------------------------------------------------
# 2. FILTER HOME / WORK
# -------------------------------------------------
user_home_hex_tampere = (
    users_tampere[users_tampere["is_home"] == 1]
    .drop_duplicates(subset="user_id")
)

user_work_hex_tampere = (
    users_tampere[users_tampere["is_work"] == 1]
    .drop_duplicates(subset="user_id")
)

# -------------------------------------------------
# 3. AGGREGATION
# -------------------------------------------------
home_counts_tampere = (
    user_home_hex_tampere.groupby("home_gid9")
    .size()
    .reset_index(name="n_users")
)

work_counts_tampere = (
    user_work_hex_tampere.groupby("work_gid9")
    .size()
    .reset_index(name="n_users")
)

# -------------------------------------------------
# 4. H3 → GEOMETRY
# -------------------------------------------------
home_counts_tampere["geometry"] = home_counts_tampere["home_gid9"].apply(h3_to_polygon)
work_counts_tampere["geometry"] = work_counts_tampere["work_gid9"].apply(h3_to_polygon)

home_gdf_tampere = gpd.GeoDataFrame(
    home_counts_tampere,
    geometry="geometry",
    crs="EPSG:4326"
)

work_gdf_tampere = gpd.GeoDataFrame(
    work_counts_tampere,
    geometry="geometry",
    crs="EPSG:4326"
)

# -------------------------------------------------
# 5. PROJECT CRS
# -------------------------------------------------
home_plot_tampere = home_gdf_tampere.to_crs(epsg=3857).copy()
work_plot_tampere = work_gdf_tampere.to_crs(epsg=3857).copy()

# -------------------------------------------------
# 6. FILTER SELECTION POLYGON
# -------------------------------------------------
filter_poly = gpd.read_parquet("./data/filter_hex_tampere.parquet").to_crs(epsg=3857)
selection_geom = filter_poly.unary_union

home_plot_tampere = home_plot_tampere[
    home_plot_tampere.geometry.centroid.within(selection_geom)
].copy()

work_plot_tampere = work_plot_tampere[
    work_plot_tampere.geometry.centroid.within(selection_geom)
].copy()

# -------------------------------------------------
# 7. HARD CAP AT 150
# -------------------------------------------------
CAP = 130

home_plot_tampere["clip"] = home_plot_tampere["n_users"].clip(upper=CAP)
work_plot_tampere["clip"] = work_plot_tampere["n_users"].clip(upper=CAP)

# -------------------------------------------------
# 8. SMOOTHING
# -------------------------------------------------
home_plot_tampere["smooth"] = gaussian_filter(home_plot_tampere["clip"].values, sigma=1)
work_plot_tampere["smooth"] = gaussian_filter(work_plot_tampere["clip"].values, sigma=1)

# -------------------------------------------------
# 9. Normalisation (fixed 0–150 scale)
# -------------------------------------------------
norm_tampere = Normalize(vmin=0, vmax=CAP)

# -------------------------------------------------
# 10. PLOT
# -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# ---------------- HOME ----------------
home_plot_tampere.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[0],
    norm=norm_tampere
)

home_plot_tampere.boundary.plot(
    ax=axes[0],
    color="darkgray",
    linewidth=0.4,
    alpha=0.9
)

ctx.add_basemap(
    axes[0],
    source=basemaps.DARK_NOLABELS
)

axes[0].set_title("Home locations (Tampere)", fontsize=11)

# ---------------- WORK ----------------
work_plot_tampere.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[1],
    norm=norm_tampere
)

work_plot_tampere.boundary.plot(
    ax=axes[1],
    color="darkgray",
    linewidth=0.4,
    alpha=0.9
)

ctx.add_basemap(
    axes[1],
    source=basemaps.DARK_NOLABELS
)

axes[1].set_title("Work locations (Tampere)", fontsize=11)

# -------------------------------------------------
# CLEAN
# -------------------------------------------------
for ax in axes:
    ax.axis("off")

plt.tight_layout()

plt.savefig(
    "./output/home_work_tampere_users_150.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter

# -------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------
users_tampere = pd.read_parquet("./data/user_pois_pt_1_tampere.parquet")

# -------------------------------------------------
# 2. FILTER HOME / WORK
# -------------------------------------------------
user_home_hex_tampere = (
    users_tampere[users_tampere["is_home"] == 1]
    .drop_duplicates(subset="user_id")
)

user_work_hex_tampere = (
    users_tampere[users_tampere["is_work"] == 1]
    .drop_duplicates(subset="user_id")
)

# -------------------------------------------------
# 3. AGGREGATION
# -------------------------------------------------
home_counts_tampere = (
    user_home_hex_tampere.groupby("home_gid9")
    .size()
    .reset_index(name="n_users")
)

work_counts_tampere = (
    user_work_hex_tampere.groupby("work_gid9")
    .size()
    .reset_index(name="n_users")
)

# -------------------------------------------------
# 4. H3 → GEOMETRY
# -------------------------------------------------
home_counts_tampere["geometry"] = home_counts_tampere["home_gid9"].apply(h3_to_polygon)
work_counts_tampere["geometry"] = work_counts_tampere["work_gid9"].apply(h3_to_polygon)

home_gdf_tampere = gpd.GeoDataFrame(
    home_counts_tampere,
    geometry="geometry",
    crs="EPSG:4326"
)

work_gdf_tampere = gpd.GeoDataFrame(
    work_counts_tampere,
    geometry="geometry",
    crs="EPSG:4326"
)

# -------------------------------------------------
# 5. PROJECT CRS
# -------------------------------------------------
home_plot_tampere = home_gdf_tampere.to_crs(epsg=3857).copy()
work_plot_tampere = work_gdf_tampere.to_crs(epsg=3857).copy()

# -------------------------------------------------
# 6. FILTER SELECTION POLYGON
# -------------------------------------------------
filter_poly = gpd.read_parquet("./data/filter_hex_tampere.parquet").to_crs(epsg=3857)
selection_geom = filter_poly.unary_union

home_plot_tampere = home_plot_tampere[
    home_plot_tampere.geometry.centroid.within(selection_geom)
].copy()

work_plot_tampere = work_plot_tampere[
    work_plot_tampere.geometry.centroid.within(selection_geom)
].copy()

# -------------------------------------------------
# 7. CLIP
# -------------------------------------------------
all_vals_tampere = np.concatenate([
    home_plot_tampere["n_users"].values,
    work_plot_tampere["n_users"].values
])

vmax_tampere = np.percentile(all_vals_tampere, 99.5)
vmin_tampere = 0

home_plot_tampere["clip"] = home_plot_tampere["n_users"].clip(upper=vmax_tampere)
work_plot_tampere["clip"] = work_plot_tampere["n_users"].clip(upper=vmax_tampere)

# -------------------------------------------------
# 8. SMOOTHING
# -------------------------------------------------
home_plot_tampere["smooth"] = gaussian_filter(home_plot_tampere["clip"].values, sigma=1)
work_plot_tampere["smooth"] = gaussian_filter(work_plot_tampere["clip"].values, sigma=1)

# -------------------------------------------------
# 9. NORMALIZATION
# -------------------------------------------------
norm_tampere = Normalize(vmin=vmin_tampere, vmax=vmax_tampere)

# -------------------------------------------------
# 10. PLOT
# -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# ---------------- HOME ----------------
home_plot_tampere.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[0],
    norm=norm_tampere
)

home_plot_tampere.boundary.plot(
    ax=axes[0],
    color="darkgray",
    linewidth=0.4,
    alpha=0.9
)

ctx.add_basemap(
    axes[0],
    source=basemaps.POSITRON_NOLABELS
)

axes[0].set_title("Home locations (Tampere)", fontsize=11)

# ---------------- WORK ----------------
work_plot_tampere.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[1],
    norm=norm_tampere
)

work_plot_tampere.boundary.plot(
    ax=axes[1],
    color="darkgray",
    linewidth=0.4,
    alpha=0.9
)

ctx.add_basemap(
    axes[1],
    source=basemaps.POSITRON_NOLABELS
)

axes[1].set_title("Work locations (Tampere)", fontsize=11)

# -------------------------------------------------
# CLEAN AXES
# -------------------------------------------------
for ax in axes:
    ax.axis("off")

# -------------------------------------------------
# SPACE FOR COLORBAR
# -------------------------------------------------
fig.subplots_adjust(right=0.88)

# -------------------------------------------------
# VERTICAL COLORBAR
# -------------------------------------------------
sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=norm_tampere)
sm._A = []

cbar_ax = fig.add_axes([0.90, 0.2, 0.02, 0.6])

cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("Number of users", fontsize=11)

# -------------------------------------------------
# SAVE FIGURE
# -------------------------------------------------
plt.savefig(
    "./output/home_work_tampere_users.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# -------------------------------------------------
# CONFIG
# -------------------------------------------------

CITY = "tampere"

file_map = {
    "pt": f"scratch/pt_typ_cat_{CITY}.parquet",
    "car": f"scratch/car_typ_cat_{CITY}.parquet",
    "bike": f"scratch/bike_typ_cat_{CITY}.parquet"
}

# -------------------------------------------------
# LOAD DATA (suffix-driven)
# -------------------------------------------------

dfs = {
    mode: pd.read_parquet(path)
    for mode, path in file_map.items()
}

# -------------------------------------------------
# FILTER ONLY JOBS
# -------------------------------------------------

jobs = {
    mode: df[df["poi_type"] == "jobs"].copy()
    for mode, df in dfs.items()
}

# -------------------------------------------------
# CLEANING RULES (centralized)
# -------------------------------------------------

def clean_series(df, col, min_val, scale=1000):
    s = df[col].dropna()
    s = s[s >= min_val] / scale
    return s

pt_vals = clean_series(jobs["pt"], "typical_trip_co2", 15)
car_vals = clean_series(jobs["car"], "typical_trip_co2", 35)
bike_vals = clean_series(jobs["bike"], "typical_trip_co2", 10)

# -------------------------------------------------
# VISUAL FILTERING (plot only)
# -------------------------------------------------

car_vis = car_vals[car_vals <= 7]
pt_vis = pt_vals.copy()
bike_vis = bike_vals.copy()

# -------------------------------------------------
# PLOT DATAFRAME
# -------------------------------------------------

plot_df = pd.concat([
    pd.DataFrame({"Mode": "Bike", "CO2": bike_vis}),
    pd.DataFrame({"Mode": "Public Transport", "CO2": pt_vis}),
    pd.DataFrame({"Mode": "Car", "CO2": car_vis})
], ignore_index=True)

# -------------------------------------------------
# STYLE
# -------------------------------------------------

sns.set_style("white")

colors = {
    "Bike": "#2b8cbe",
    "Public Transport": "#7bccc4",
    "Car": "#de2d26"
}

mode_order = ["Bike", "Public Transport", "Car"]

# -------------------------------------------------
# FIGURE
# -------------------------------------------------

fig, ax = plt.subplots(figsize=(8.5, 4.5))

sns.violinplot(
    data=plot_df,
    y="Mode",
    x="CO2",
    order=mode_order,
    orient="h",
    palette=colors,
    inner=None,
    linewidth=0,
    cut=0,
    bw_adjust=0.9,
    saturation=1,
    width=0.9,
    ax=ax
)

# soften violins
for c in ax.collections:
    c.set_alpha(0.38)

# -------------------------------------------------
# MEDIANS (true values)
# -------------------------------------------------

medians = {
    "Bike": bike_vals.median(),
    "Public Transport": pt_vals.median(),
    "Car": car_vals.median()
}

for i, mode in enumerate(mode_order):
    m = medians[mode]
    x = min(m, 7)

    ax.scatter(x, i, s=95, color="black", zorder=10)

    ax.text(
        x + 0.10,
        i,
        f"{m:.2f} kg",
        va="center",
        ha="left",
        fontsize=9,
        fontweight="bold"
    )

# -------------------------------------------------
# AXES
# -------------------------------------------------

ax.set_xlim(0, 7)
ax.set_xticks(np.arange(0, 8, 1))
ax.set_xlabel("Commuting CO₂ per trip (kg)")
ax.set_ylabel("")
ax.set_title(f"{CITY.title()}: CO₂ distribution by transport mode", pad=14)

ax.xaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.22)
ax.yaxis.grid(False)

sns.despine(left=False, bottom=False)

plt.tight_layout()

# -------------------------------------------------
# SAVE
# -------------------------------------------------

plt.savefig(
    f"{CITY}_commuting_co2_violin_jobs.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# -------------------------------------------------
# SUMMARY
# -------------------------------------------------

print("Observations used in visualization")
print("----------------------------------")
for mode in ["bike", "pt", "car"]:
    print(f"{mode.upper()}: {len(jobs[mode][jobs[mode]['typical_trip_co2'].notna()]):,}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------
# LOAD DATA (TAMPERE)
# -------------------------------------------------
df_jobs_bike_tampere = pd.read_parquet(
    "scratch/bike_typ_cat_tampere.parquet"
)

df_jobs_car_tampere = pd.read_parquet(
    "scratch/car_typ_cat_tampere.parquet"
)

df_jobs_tampere = pd.read_parquet(
    "scratch/pt_typ_cat_tampere.parquet"
)

# -------------------------------------------------
# PURPOSE WEIGHTS
# -------------------------------------------------
purpose_weights = {
    "jobs": 4,
    "Recreational, Outdoors": 2,
    "Social, Cultural": 2,
    "Shopping, Errands": 1
}

# -------------------------------------------------
# MEDIAN-BASED COMPONENTS
# -------------------------------------------------
def compute_components(df, mode):

    rows = []

    for purpose, weight in purpose_weights.items():

        subset = df[df["poi_type"] == purpose]["typical_trip_co2"].dropna()
        subset = subset / 1000  # g → kg

        value = subset.median()

        rows.append({
            "Mode": mode,
            "Purpose": purpose,
            "Value": value * weight
        })

    return pd.DataFrame(rows)

# -------------------------------------------------
# BUILD DATA
# -------------------------------------------------
bike = compute_components(df_jobs_bike_tampere, "Bike")
pt   = compute_components(df_jobs_tampere, "Public Transport")
car  = compute_components(df_jobs_car_tampere, "Car")

df = pd.concat([bike, pt, car])

# -------------------------------------------------
# PIVOT
# -------------------------------------------------
pivot = df.pivot_table(
    index="Mode",
    columns="Purpose",
    values="Value",
    aggfunc="sum"
).fillna(0)

order = ["Bike", "Public Transport", "Car"]
pivot = pivot.loc[order]

# -------------------------------------------------
# SPLIT
# -------------------------------------------------
commuting = pivot["jobs"]
non_work = pivot.drop(columns=["jobs"]).sum(axis=1)

total = commuting + non_work

commuting_pct = (commuting / total) * 100
non_work_pct = (non_work / total) * 100

# -------------------------------------------------
# COLORS
# -------------------------------------------------
colors = {
    "commuting": "#1f2a44",
    "non_work": "#4c78a8"
}

# -------------------------------------------------
# FIGURE
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 4.4))

y = np.arange(len(order))
bar_h = 0.38

# bars
ax.barh(y, commuting_pct.values, height=bar_h, color=colors["commuting"])
ax.barh(
    y,
    non_work_pct.values,
    left=commuting_pct.values,
    height=bar_h,
    color=colors["non_work"]
)

# -------------------------------------------------
# LABELS
# -------------------------------------------------
for i in range(len(order)):

    c = commuting_pct.iloc[i]
    n = non_work_pct.iloc[i]
    t = total.iloc[i]

    ax.text(c / 2, i, f"{c:.0f}%",
            ha="center", va="center",
            color="white", fontweight="bold")

    ax.text(c + n / 2, i, f"{n:.0f}%",
            ha="center", va="center",
            color="white", fontweight="bold")

    ax.text(102, i, f"Total: {t:.2f} kg CO₂",
            va="center", fontsize=10, fontweight="bold")

# -------------------------------------------------
# FORMATTING
# -------------------------------------------------
ax.set_yticks(y)
ax.set_yticklabels(order)

ax.set_xlim(0, 110)
ax.set_xlabel("Share of CO₂ footprint per trip purpose (%)")

ax.set_title(
    "Decent mobility CO₂ composition (Tampere)\nPer-user median-based index",
    pad=14
)

ax.xaxis.grid(True, linestyle="--", alpha=0.25)
ax.yaxis.grid(False)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

# -------------------------------------------------
# Legend below the plot
# -------------------------------------------------
ax.legend(
    ["Commuting (jobs)", "Non-work trips"],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.28),
    ncol=2,
    frameon=False
)

plt.subplots_adjust(bottom=0.25, top=0.88)
plt.tight_layout()

# -------------------------------------------------
# SAVE
# -------------------------------------------------
plt.savefig(
    "decent_mobility_composition_tampere_FIXED.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter

# -------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------
users_oulu = pd.read_parquet("./data/user_pois_pt_1_oulu.parquet")

# -------------------------------------------------
# 2. FILTER HOME / WORK
# -------------------------------------------------
user_home_hex_oulu = (
    users_oulu[users_oulu["is_home"] == 1]
    .drop_duplicates(subset="user_id")
)

user_work_hex_oulu = (
    users_oulu[users_oulu["is_work"] == 1]
    .drop_duplicates(subset="user_id")
)

# -------------------------------------------------
# 3. AGGREGATION
# -------------------------------------------------
home_counts_oulu = (
    user_home_hex_oulu.groupby("home_gid9")
    .size()
    .reset_index(name="n_users")
)

work_counts_oulu = (
    user_work_hex_oulu.groupby("work_gid9")
    .size()
    .reset_index(name="n_users")
)

# -------------------------------------------------
# 4. H3 → GEOMETRY
# -------------------------------------------------
home_counts_oulu["geometry"] = home_counts_oulu["home_gid9"].apply(h3_to_polygon)
work_counts_oulu["geometry"] = work_counts_oulu["work_gid9"].apply(h3_to_polygon)

home_gdf_oulu = gpd.GeoDataFrame(
    home_counts_oulu,
    geometry="geometry",
    crs="EPSG:4326"
)

work_gdf_oulu = gpd.GeoDataFrame(
    work_counts_oulu,
    geometry="geometry",
    crs="EPSG:4326"
)

# -------------------------------------------------
# 5. PROJECT CRS
# -------------------------------------------------
home_plot_oulu = home_gdf_oulu.to_crs(epsg=3857).copy()
work_plot_oulu = work_gdf_oulu.to_crs(epsg=3857).copy()

# -------------------------------------------------
# 6. LOAD SELECTION POLYGON + FILTER
# -------------------------------------------------
filter_poly = gpd.read_parquet("./data/filter_hex_oulu.parquet").to_crs(epsg=3857)
selection_geom = filter_poly.unary_union

home_plot_oulu = home_plot_oulu[
    home_plot_oulu.geometry.centroid.within(selection_geom)
].copy()

work_plot_oulu = work_plot_oulu[
    work_plot_oulu.geometry.centroid.within(selection_geom)
].copy()

# -------------------------------------------------
# 7. HARD CAP AT 150 (IMPORTANT CHANGE)
# -------------------------------------------------
CAP = 120

home_plot_oulu["clip"] = home_plot_oulu["n_users"].clip(upper=CAP)
work_plot_oulu["clip"] = work_plot_oulu["n_users"].clip(upper=CAP)

# -------------------------------------------------
# 8. SMOOTHING
# -------------------------------------------------
home_plot_oulu["smooth"] = gaussian_filter(home_plot_oulu["clip"].values, sigma=1)
work_plot_oulu["smooth"] = gaussian_filter(work_plot_oulu["clip"].values, sigma=1)

# -------------------------------------------------
# 9. Normalisation (fixed 0–150 scale)
# -------------------------------------------------
norm_oulu = Normalize(vmin=0, vmax=CAP)

# -------------------------------------------------
# 10. PLOT
# -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# ---------------- HOME ----------------
home_plot_oulu.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[0],
    norm=norm_oulu
)

home_plot_oulu.boundary.plot(
    ax=axes[0],
    color="darkgray",
    linewidth=0.4,
    alpha=0.9
)

ctx.add_basemap(
    axes[0],
    source=basemaps.DARK_NOLABELS
)

axes[0].set_title("Home locations (Oulu)", fontsize=11)

# ---------------- WORK ----------------
work_plot_oulu.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[1],
    norm=norm_oulu
)

work_plot_oulu.boundary.plot(
    ax=axes[1],
    color="darkgray",
    linewidth=0.4,
    alpha=0.9
)

ctx.add_basemap(
    axes[1],
    source=basemaps.DARK_NOLABELS
)

axes[1].set_title("Work locations (Oulu)", fontsize=11)

# -------------------------------------------------
# CLEAN
# -------------------------------------------------
for ax in axes:
    ax.axis("off")

plt.tight_layout()

plt.savefig(
    "./output/home_work_oulu_users_150.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter

# -------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------
users_oulu = pd.read_parquet("./data/user_pois_pt_1_oulu.parquet")

# -------------------------------------------------
# 2. FILTER HOME / WORK
# -------------------------------------------------
user_home_hex_oulu = (
    users_oulu[users_oulu["is_home"] == 1]
    .drop_duplicates(subset="user_id")
)

user_work_hex_oulu = (
    users_oulu[users_oulu["is_work"] == 1]
    .drop_duplicates(subset="user_id")
)

# -------------------------------------------------
# 3. AGGREGATION
# -------------------------------------------------
home_counts_oulu = (
    user_home_hex_oulu.groupby("home_gid9")
    .size()
    .reset_index(name="n_users")
)

work_counts_oulu = (
    user_work_hex_oulu.groupby("work_gid9")
    .size()
    .reset_index(name="n_users")
)

# -------------------------------------------------
# 4. H3 → GEOMETRY
# -------------------------------------------------
home_counts_oulu["geometry"] = home_counts_oulu["home_gid9"].apply(h3_to_polygon)
work_counts_oulu["geometry"] = work_counts_oulu["work_gid9"].apply(h3_to_polygon)

home_gdf_oulu = gpd.GeoDataFrame(
    home_counts_oulu,
    geometry="geometry",
    crs="EPSG:4326"
)

work_gdf_oulu = gpd.GeoDataFrame(
    work_counts_oulu,
    geometry="geometry",
    crs="EPSG:4326"
)

# -------------------------------------------------
# 5. PROJECT CRS
# -------------------------------------------------
home_plot_oulu = home_gdf_oulu.to_crs(epsg=3857).copy()
work_plot_oulu = work_gdf_oulu.to_crs(epsg=3857).copy()

# -------------------------------------------------
# 6. FILTER SELECTION POLYGON
# -------------------------------------------------
filter_poly = gpd.read_parquet("./data/filter_hex_oulu.parquet").to_crs(epsg=3857)
selection_geom = filter_poly.unary_union

home_plot_oulu = home_plot_oulu[
    home_plot_oulu.geometry.centroid.within(selection_geom)
].copy()

work_plot_oulu = work_plot_oulu[
    work_plot_oulu.geometry.centroid.within(selection_geom)
].copy()

# -------------------------------------------------
# 7. CLIP
# -------------------------------------------------
all_vals_oulu = np.concatenate([
    home_plot_oulu["n_users"].values,
    work_plot_oulu["n_users"].values
])

vmax_oulu = np.percentile(all_vals_oulu, 99.5)
vmin_oulu = 0

home_plot_oulu["clip"] = home_plot_oulu["n_users"].clip(upper=vmax_oulu)
work_plot_oulu["clip"] = work_plot_oulu["n_users"].clip(upper=vmax_oulu)

# -------------------------------------------------
# 8. SMOOTHING
# -------------------------------------------------
home_plot_oulu["smooth"] = gaussian_filter(home_plot_oulu["clip"].values, sigma=1)
work_plot_oulu["smooth"] = gaussian_filter(work_plot_oulu["clip"].values, sigma=1)

# -------------------------------------------------
# 9. NORMALIZATION
# -------------------------------------------------
norm_oulu = Normalize(vmin=vmin_oulu, vmax=vmax_oulu)

# -------------------------------------------------
# 10. PLOT
# -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# ---------------- HOME ----------------
home_plot_oulu.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[0],
    norm=norm_oulu
)

home_plot_oulu.boundary.plot(
    ax=axes[0],
    color="darkgray",
    linewidth=0.4,
    alpha=0.9
)

ctx.add_basemap(
    axes[0],
    source=basemaps.POSITRON_NOLABELS
)

axes[0].set_title("Home locations (Oulu)", fontsize=11)

# ---------------- WORK ----------------
work_plot_oulu.plot(
    column="smooth",
    cmap="YlOrRd",
    linewidth=0,
    alpha=0.99,
    ax=axes[1],
    norm=norm_oulu
)

work_plot_oulu.boundary.plot(
    ax=axes[1],
    color="darkgray",
    linewidth=0.4,
    alpha=0.9
)

ctx.add_basemap(
    axes[1],
    source=basemaps.POSITRON_NOLABELS
)

axes[1].set_title("Work locations (Oulu)", fontsize=11)

# -------------------------------------------------
# CLEAN AXES
# -------------------------------------------------
for ax in axes:
    ax.axis("off")

# -------------------------------------------------
# SPACE FOR COLORBAR
# -------------------------------------------------
fig.subplots_adjust(right=0.88)

# -------------------------------------------------
# VERTICAL COLORBAR
# -------------------------------------------------
sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=norm_oulu)
sm._A = []

cbar_ax = fig.add_axes([0.90, 0.2, 0.02, 0.6])

cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("Number of users", fontsize=11)

# -------------------------------------------------
# SAVE FIGURE
# -------------------------------------------------
plt.savefig(
    "./output/home_work_oulu_users.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# -------------------------------------------------
# CONFIG
# -------------------------------------------------

CITY = "oulu"

file_map = {
    "pt": f"scratch/pt_typ_cat_{CITY}.parquet",
    "car": f"scratch/car_typ_cat_{CITY}.parquet",
    "bike": f"scratch/bike_typ_cat_{CITY}.parquet"
}

# -------------------------------------------------
# LOAD DATA (suffix-driven)
# -------------------------------------------------

dfs = {
    mode: pd.read_parquet(path)
    for mode, path in file_map.items()
}

# -------------------------------------------------
# FILTER ONLY JOBS
# -------------------------------------------------

jobs = {
    mode: df[df["poi_type"] == "jobs"].copy()
    for mode, df in dfs.items()
}

# -------------------------------------------------
# CLEANING RULES (centralized)
# -------------------------------------------------

def clean_series(df, col, min_val, scale=1000):
    s = df[col].dropna()
    s = s[s >= min_val] / scale
    return s

pt_vals = clean_series(jobs["pt"], "typical_trip_co2", 15)
car_vals = clean_series(jobs["car"], "typical_trip_co2", 35)
bike_vals = clean_series(jobs["bike"], "typical_trip_co2", 10)

# -------------------------------------------------
# VISUAL FILTERING (plot only)
# -------------------------------------------------

car_vis = car_vals[car_vals <= 7]
pt_vis = pt_vals.copy()
bike_vis = bike_vals.copy()

# -------------------------------------------------
# PLOT DATAFRAME
# -------------------------------------------------

plot_df = pd.concat([
    pd.DataFrame({"Mode": "Bike", "CO2": bike_vis}),
    pd.DataFrame({"Mode": "Public Transport", "CO2": pt_vis}),
    pd.DataFrame({"Mode": "Car", "CO2": car_vis})
], ignore_index=True)

# -------------------------------------------------
# STYLE
# -------------------------------------------------

sns.set_style("white")

colors = {
    "Bike": "#2b8cbe",
    "Public Transport": "#7bccc4",
    "Car": "#de2d26"
}

mode_order = ["Bike", "Public Transport", "Car"]

# -------------------------------------------------
# FIGURE
# -------------------------------------------------

fig, ax = plt.subplots(figsize=(8.5, 4.5))

sns.violinplot(
    data=plot_df,
    y="Mode",
    x="CO2",
    order=mode_order,
    orient="h",
    palette=colors,
    inner=None,
    linewidth=0,
    cut=0,
    bw_adjust=0.9,
    saturation=1,
    width=0.9,
    ax=ax
)

# soften violins
for c in ax.collections:
    c.set_alpha(0.38)

# -------------------------------------------------
# MEDIANS (true values)
# -------------------------------------------------

medians = {
    "Bike": bike_vals.median(),
    "Public Transport": pt_vals.median(),
    "Car": car_vals.median()
}

for i, mode in enumerate(mode_order):
    m = medians[mode]
    x = min(m, 7)

    ax.scatter(x, i, s=95, color="black", zorder=10)

    ax.text(
        x + 0.10,
        i,
        f"{m:.2f} kg",
        va="center",
        ha="left",
        fontsize=9,
        fontweight="bold"
    )

# -------------------------------------------------
# AXES
# -------------------------------------------------

ax.set_xlim(0, 7)
ax.set_xticks(np.arange(0, 8, 1))
ax.set_xlabel("Commuting CO₂ per trip (kg)")
ax.set_ylabel("")
ax.set_title(f"{CITY.title()}: CO₂ distribution by transport mode", pad=14)

ax.xaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.22)
ax.yaxis.grid(False)

sns.despine(left=False, bottom=False)

plt.tight_layout()

# -------------------------------------------------
# SAVE
# -------------------------------------------------

plt.savefig(
    f"{CITY}_commuting_co2_violin_jobs.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# -------------------------------------------------
# SUMMARY
# -------------------------------------------------

print("Observations used in visualization")
print("----------------------------------")
for mode in ["bike", "pt", "car"]:
    print(f"{mode.upper()}: {len(jobs[mode][jobs[mode]['typical_trip_co2'].notna()]):,}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------
# LOAD DATA (OULU)
# -------------------------------------------------
df_jobs_bike_oulu = pd.read_parquet(
    "scratch/bike_typ_cat_oulu.parquet"
)

df_jobs_car_oulu = pd.read_parquet(
    "scratch/car_typ_cat_oulu.parquet"
)

df_jobs_oulu = pd.read_parquet(
    "scratch/pt_typ_cat_oulu.parquet"
)

# -------------------------------------------------
# PURPOSE WEIGHTS
# -------------------------------------------------
purpose_weights = {
    "jobs": 4,
    "Recreational, Outdoors": 2,
    "Social, Cultural": 2,
    "Shopping, Errands": 1
}

# -------------------------------------------------
# MEDIAN-BASED COMPONENTS
# -------------------------------------------------
def compute_components(df, mode):

    rows = []

    for purpose, weight in purpose_weights.items():

        subset = df[df["poi_type"] == purpose]["typical_trip_co2"].dropna()
        subset = subset / 1000  # g → kg

        value = subset.median()

        rows.append({
            "Mode": mode,
            "Purpose": purpose,
            "Value": value * weight
        })

    return pd.DataFrame(rows)

# -------------------------------------------------
# BUILD DATA
# -------------------------------------------------
bike = compute_components(df_jobs_bike_oulu, "Bike")
pt   = compute_components(df_jobs_oulu, "Public Transport")
car  = compute_components(df_jobs_car_oulu, "Car")

df = pd.concat([bike, pt, car])

# -------------------------------------------------
# PIVOT
# -------------------------------------------------
pivot = df.pivot_table(
    index="Mode",
    columns="Purpose",
    values="Value",
    aggfunc="sum"
).fillna(0)

order = ["Bike", "Public Transport", "Car"]
pivot = pivot.loc[order]

# -------------------------------------------------
# SPLIT
# -------------------------------------------------
commuting = pivot["jobs"]
non_work = pivot.drop(columns=["jobs"]).sum(axis=1)

total = commuting + non_work

commuting_pct = (commuting / total) * 100
non_work_pct = (non_work / total) * 100

# -------------------------------------------------
# COLORS
# -------------------------------------------------
colors = {
    "commuting": "#1f2a44",
    "non_work": "#4c78a8"
}

# -------------------------------------------------
# FIGURE
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 4.4))

y = np.arange(len(order))
bar_h = 0.38

# bars
ax.barh(y, commuting_pct.values, height=bar_h, color=colors["commuting"])
ax.barh(
    y,
    non_work_pct.values,
    left=commuting_pct.values,
    height=bar_h,
    color=colors["non_work"]
)

# -------------------------------------------------
# LABELS
# -------------------------------------------------
for i in range(len(order)):

    c = commuting_pct.iloc[i]
    n = non_work_pct.iloc[i]
    t = total.iloc[i]

    ax.text(c / 2, i, f"{c:.0f}%",
            ha="center", va="center",
            color="white", fontweight="bold")

    ax.text(c + n / 2, i, f"{n:.0f}%",
            ha="center", va="center",
            color="white", fontweight="bold")

    ax.text(102, i, f"Total: {t:.2f} kg CO₂",
            va="center", fontsize=10, fontweight="bold")

# -------------------------------------------------
# FORMATTING
# -------------------------------------------------
ax.set_yticks(y)
ax.set_yticklabels(order)

ax.set_xlim(0, 110)
ax.set_xlabel("Share of CO₂ footprint per trip purpose (%)")

ax.set_title(
    "Decent mobility CO₂ composition (Oulu)\nPer-user median-based index",
    pad=14
)

ax.xaxis.grid(True, linestyle="--", alpha=0.25)
ax.yaxis.grid(False)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

# -------------------------------------------------
# Legend below the plot
# -------------------------------------------------
ax.legend(
    ["Commuting (jobs)", "Non-work trips"],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.28),
    ncol=2,
    frameon=False
)

plt.subplots_adjust(bottom=0.25, top=0.88)
plt.tight_layout()

# -------------------------------------------------
# SAVE
# -------------------------------------------------
plt.savefig(
    "decent_mobility_composition_oulu_FIXED.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### National


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =================================================
# LOAD ALL CITY DATA
# =================================================

df_jobs_bike_fi = pd.concat([
    df_jobs_bike,
    df_jobs_bike_turku,
    df_jobs_bike_tampere,
    df_jobs_bike_oulu
])

df_jobs_car_fi = pd.concat([
    df_jobs_cars,
    df_jobs_car_turku,
    df_jobs_car_tampere,
    df_jobs_car_oulu
])

df_jobs_pt_fi = pd.concat([
    df_jobs,
    df_jobs_turku,
    df_jobs_tampere,
    df_jobs_oulu
])

# =================================================
# PURPOSE WEIGHTS
# =================================================

purpose_weights = {
    "jobs": 4,
    "Recreational, Outdoors": 2,
    "Social, Cultural": 2,
    "Shopping, Errands": 1
}

required_purposes = list(purpose_weights.keys())

# =================================================
# PER-USER INDEX (STRICT, NO IMPUTATION)
# =================================================

def build_user_index(df):

    df = df.copy()
    df["co2_kg"] = df["typical_trip_co2"] / 1000

    pivot = df.pivot_table(
        index="user_id",
        columns="poi_type",
        values="co2_kg",
        aggfunc="first"
    )

    # keep only complete users (IMPORTANT)
    pivot = pivot.dropna(subset=required_purposes)

    # apply weights
    for c in required_purposes:
        pivot[c] = pivot[c] * purpose_weights[c]

    total = pivot.sum(axis=1)
    commuting = pivot["jobs"]
    non_work = total - commuting

    return commuting, non_work

# =================================================
# COMPUTE NATIONAL DISTRIBUTIONS
# =================================================

bike_comm, bike_non = build_user_index(df_jobs_bike_fi)
pt_comm, pt_non     = build_user_index(df_jobs_pt_fi)
car_comm, car_non   = build_user_index(df_jobs_car_fi)

# =================================================
# MEDIANS (CORE METRIC)
# =================================================

order = ["Bike", "Public Transport", "Car"]

data = pd.DataFrame({
    "Mode": order,
    "Commuting": [
        bike_comm.median(),
        pt_comm.median(),
        car_comm.median()
    ],
    "NonWork": [
        bike_non.median(),
        pt_non.median(),
        car_non.median()
    ]
})

# =================================================
# SHARES
# =================================================

data["Total"] = data["Commuting"] + data["NonWork"]
data["Commuting_pct"] = data["Commuting"] / data["Total"] * 100
data["NonWork_pct"] = data["NonWork"] / data["Total"] * 100

# =================================================
# PLOT
# =================================================

colors = {
    "commuting": "#1f2a44",
    "non_work": "#4c78a8"
}

fig, ax = plt.subplots(figsize=(11, 4.4))

y = np.arange(len(order))
bar_h = 0.45

ax.barh(y, data["Commuting_pct"], height=bar_h, color=colors["commuting"])
ax.barh(
    y,
    data["NonWork_pct"],
    left=data["Commuting_pct"],
    height=bar_h,
    color=colors["non_work"]
)

# =================================================
# LABELS
# =================================================

for i in range(len(order)):

    c = data["Commuting_pct"].iloc[i]
    n = data["NonWork_pct"].iloc[i]
    t = data["Total"].iloc[i]

    ax.text(c/2, i, f"{c:.0f}%",
            ha="center", va="center",
            color="white", fontweight="bold")

    ax.text(c + n/2, i, f"{n:.0f}%",
            ha="center", va="center",
            color="white", fontweight="bold")

    ax.text(102, i, f"Total: {t:.2f} kg CO₂",
            va="center", fontsize=10, fontweight="bold")

# =================================================
# FORMATTING
# =================================================

ax.set_yticks(y)
ax.set_yticklabels(order)

ax.set_xlim(0, 110)
ax.set_xlabel("Share of CO₂ footprint per trip purpose (%)")

ax.set_title(
    "Decent mobility CO₂ composition (Finland)\nPer-user median-based index",
    pad=12
)

ax.xaxis.grid(True, linestyle="--", alpha=0.25)
ax.yaxis.grid(False)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

# =================================================
# Legend below the plot
# =================================================

ax.legend(
    ["Commuting (jobs)", "Non-work trips"],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=2,
    frameon=False
)

plt.tight_layout()

plt.savefig(
    "decent_mobility_composition_finland_FIXED.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
car_comm

In [ ]:
car_non

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =================================================
# LOAD ALL CITY DATA
# =================================================

df_jobs_bike_fi = pd.concat([
    df_jobs_bike,
    df_jobs_bike_turku,
    df_jobs_bike_tampere,
    df_jobs_bike_oulu
])

df_jobs_car_fi = pd.concat([
    df_jobs_cars,
    df_jobs_car_turku,
    df_jobs_car_tampere,
    df_jobs_car_oulu
])

df_jobs_pt_fi = pd.concat([
    df_jobs,
    df_jobs_turku,
    df_jobs_tampere,
    df_jobs_oulu
])

# =================================================
# REMOVE VERY SMALL CAR VALUES (<0.2 kg)
# =================================================

df_jobs_car_fi = df_jobs_car_fi.copy()

df_jobs_car_fi["typical_trip_co2_kg"] = (
    df_jobs_car_fi["typical_trip_co2"] / 1000
)

df_jobs_car_fi = df_jobs_car_fi[
    df_jobs_car_fi["typical_trip_co2_kg"] >= 0.2
]

# =================================================
# PURPOSE WEIGHTS
# =================================================

purpose_weights = {
    "jobs": 4,
    "Recreational, Outdoors": 2,
    "Social, Cultural": 2,
    "Shopping, Errands": 1
}

required_purposes = list(purpose_weights.keys())

# =================================================
# PER-USER INDEX (STRICT, NO IMPUTATION)
# =================================================

def build_user_index(df):

    df = df.copy()

    # convert to kg
    if "typical_trip_co2_kg" not in df.columns:
        df["co2_kg"] = df["typical_trip_co2"] / 1000
    else:
        df["co2_kg"] = df["typical_trip_co2_kg"]

    # user × purpose matrix
    pivot = df.pivot_table(
        index="user_id",
        columns="poi_type",
        values="co2_kg",
        aggfunc="first"
    )

    # STRICT COMPLETE USERS ONLY
    pivot = pivot.dropna(subset=required_purposes)

    # apply purpose weights
    for c in required_purposes:
        pivot[c] = pivot[c] * purpose_weights[c]

    total = pivot.sum(axis=1)

    commuting = pivot["jobs"]
    non_work = total - commuting

    return commuting, non_work

# =================================================
# COMPUTE NATIONAL DISTRIBUTIONS
# =================================================

bike_comm, bike_non = build_user_index(df_jobs_bike_fi)
pt_comm, pt_non     = build_user_index(df_jobs_pt_fi)
car_comm, car_non   = build_user_index(df_jobs_car_fi)

# =================================================
# MEDIANS
# =================================================

order = ["Bike", "Public Transport", "Car"]

data = pd.DataFrame({
    "Mode": order,
    "Commuting": [
        bike_comm.median(),
        pt_comm.median(),
        car_comm.median()
    ],
    "NonWork": [
        bike_non.median(),
        pt_non.median(),
        car_non.median()
    ]
})

# =================================================
# SHARES
# =================================================

data["Total"] = data["Commuting"] + data["NonWork"]

data["Commuting_pct"] = (
    data["Commuting"] / data["Total"] * 100
)

data["NonWork_pct"] = (
    data["NonWork"] / data["Total"] * 100
)

# =================================================
# PLOT
# =================================================

colors = {
    "commuting": "#1f2a44",
    "non_work": "#4c78a8"
}

fig, ax = plt.subplots(figsize=(11, 4.4))

y = np.arange(len(order))
bar_h = 0.45

# commuting
ax.barh(
    y,
    data["Commuting_pct"],
    height=bar_h,
    color=colors["commuting"]
)

# non-work
ax.barh(
    y,
    data["NonWork_pct"],
    left=data["Commuting_pct"],
    height=bar_h,
    color=colors["non_work"]
)

# =================================================
# LABELS
# =================================================

for i in range(len(order)):

    c = data["Commuting_pct"].iloc[i]
    n = data["NonWork_pct"].iloc[i]
    t = data["Total"].iloc[i]

    ax.text(
        c / 2,
        i,
        f"{c:.0f}%",
        ha="center",
        va="center",
        color="white",
        fontweight="bold"
    )

    ax.text(
        c + n / 2,
        i,
        f"{n:.0f}%",
        ha="center",
        va="center",
        color="white",
        fontweight="bold"
    )

    ax.text(
        102,
        i,
        f"Total: {t:.2f} kg CO₂",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

# =================================================
# FORMATTING
# =================================================

ax.set_yticks(y)
ax.set_yticklabels(order)

ax.set_xlim(0, 110)

ax.set_xlabel(
    "Share of CO₂ footprint per trip purpose (%)"
)

ax.set_title(
    "Decent mobility CO₂ composition (Finland)\nPer-user median-based index",
    pad=14
)

ax.xaxis.grid(True, linestyle="--", alpha=0.25)
ax.yaxis.grid(False)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

# =================================================
# LEGEND
# =================================================

ax.legend(
    ["Commuting (jobs)", "Non-work trips"],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=2,
    frameon=False
)

plt.tight_layout()

plt.savefig(
    "decent_mobility_composition_finland_filtered.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =================================================
# CLEAN VALUES
# =================================================

car_comm_clean = car_comm.dropna()
car_non_clean = car_non.dropna()

# medians
median_comm = car_comm_clean.median()
median_non = car_non_clean.median()

# =================================================
# FIGURE
# =================================================

fig, ax = plt.subplots(figsize=(10.5, 5))

# -------------------------------------------------
# HISTOGRAMS
# -------------------------------------------------

bins = 40

ax.hist(
    car_comm_clean,
    bins=bins,
    alpha=0.55,
    density=True,
    label="Commuting (jobs)",
    color="#1f2a44"
)

ax.hist(
    car_non_clean,
    bins=bins,
    alpha=0.45,
    density=True,
    label="Non-work trips",
    color="#4c78a8"
)

# -------------------------------------------------
# MEDIAN LINES
# -------------------------------------------------

ax.axvline(
    median_comm,
    linestyle="--",
    linewidth=2.2,
    color="#1f2a44"
)

ax.axvline(
    median_non,
    linestyle="--",
    linewidth=2.2,
    color="#4c78a8"
)

# -------------------------------------------------
# MEDIAN LABELS
# -------------------------------------------------

ylim = ax.get_ylim()[1]

ax.text(
    median_comm,
    ylim * 0.92,
    f"Median commuting\n{median_comm:.2f} kg",
    color="#1f2a44",
    fontsize=10,
    fontweight="bold",
    ha="left"
)

ax.text(
    median_non,
    ylim * 0.72,
    f"Median non-work\n{median_non:.2f} kg",
    color="#4c78a8",
    fontsize=10,
    fontweight="bold",
    ha="left"
)

# =================================================
# FORMATTING
# =================================================

ax.set_xlabel("Weekly CO₂ footprint contribution (kg CO₂)")
ax.set_ylabel("Density")

ax.set_title(
    "Distribution of commuting and non-work CO₂ contributions\nCar users (Finland)",
    pad=14
)

ax.grid(axis="y", linestyle="--", alpha=0.2)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

ax.legend(
    frameon=False,
    loc="upper right"
)

plt.tight_layout()

plt.savefig(
    "distribution_car_commuting_nonwork.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =================================================
# USER-LEVEL SHARES
# =================================================

# total per user
car_total = car_comm + car_non

# user-level percentages
car_comm_share = (car_comm / car_total) * 100
car_non_share = (car_non / car_total) * 100

# remove impossible values
mask = (
    car_comm_share.notna() &
    car_non_share.notna() &
    np.isfinite(car_comm_share) &
    np.isfinite(car_non_share)
)

car_comm_share = car_comm_share[mask]
car_non_share = car_non_share[mask]

# =================================================
# MEDIANS OF SHARES
# =================================================

median_comm_share = car_comm_share.median()
median_non_share = car_non_share.median()

print(f"Median commuting share: {median_comm_share:.2f}%")
print(f"Median non-work share: {median_non_share:.2f}%")

# =================================================
# PLOT
# =================================================

fig, ax = plt.subplots(figsize=(10.5, 5))

bins = np.linspace(0, 100, 40)

# commuting share distribution
ax.hist(
    car_comm_share,
    bins=bins,
    density=True,
    alpha=0.55,
    color="#1f2a44",
    label="Commuting share"
)

# non-work share distribution
ax.hist(
    car_non_share,
    bins=bins,
    density=True,
    alpha=0.45,
    color="#4c78a8",
    label="Non-work share"
)

# =================================================
# MEDIAN LINES
# =================================================

ax.axvline(
    median_comm_share,
    linestyle="--",
    linewidth=2.2,
    color="#1f2a44"
)

ax.axvline(
    median_non_share,
    linestyle="--",
    linewidth=2.2,
    color="#4c78a8"
)

# =================================================
# LABELS
# =================================================

ylim = ax.get_ylim()[1]

ax.text(
    median_comm_share,
    ylim * 0.92,
    f"Median commuting\n{median_comm_share:.1f}%",
    color="#1f2a44",
    fontsize=10,
    fontweight="bold",
    ha="left"
)

ax.text(
    median_non_share,
    ylim * 0.72,
    f"Median non-work\n{median_non_share:.1f}%",
    color="#4c78a8",
    fontsize=10,
    fontweight="bold",
    ha="right"
)

# =================================================
# FORMATTING
# =================================================

ax.set_xlim(0, 100)

ax.set_xlabel("Share of weekly CO₂ footprint (%)")
ax.set_ylabel("Density")

ax.set_title(
    "Distribution of commuting and non-work CO₂ shares\nCar users (Finland)",
    pad=14
)

ax.grid(axis="y", linestyle="--", alpha=0.2)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

ax.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=2
)

plt.tight_layout()

plt.savefig(
    "distribution_car_commuting_shares.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =================================================
# LOAD ALL CITY DATA
# =================================================

df_jobs_bike_fi = pd.concat([
    df_jobs_bike,
    df_jobs_bike_turku,
    df_jobs_bike_tampere,
    df_jobs_bike_oulu
])

df_jobs_car_fi = pd.concat([
    df_jobs_cars,
    df_jobs_car_turku,
    df_jobs_car_tampere,
    df_jobs_car_oulu
])

df_jobs_pt_fi = pd.concat([
    df_jobs,
    df_jobs_turku,
    df_jobs_tampere,
    df_jobs_oulu
])

# =================================================
# PURPOSE WEIGHTS
# =================================================

purpose_weights = {
    "jobs": 4,
    "Recreational, Outdoors": 2,
    "Social, Cultural": 2,
    "Shopping, Errands": 1
}

required_purposes = list(purpose_weights.keys())

# =================================================
# PER-USER SHARES + TOTALS
# =================================================

def build_user_shares(df):

    df = df.copy()
    df["co2_kg"] = df["typical_trip_co2"] / 1000

    pivot = df.pivot_table(
        index="user_id",
        columns="poi_type",
        values="co2_kg",
        aggfunc="first"
    )

    # keep only complete users
    pivot = pivot.dropna(subset=required_purposes)

    # apply weights
    for c in required_purposes:
        pivot[c] = pivot[c] * purpose_weights[c]

    total = pivot.sum(axis=1)
    commuting = pivot["jobs"]
    non_work = total - commuting

    comm_share = commuting / total
    non_share = non_work / total

    # ================================
    # SANITY FILTER (KEY FIX)
    # ================================
    valid_mask = (
        comm_share.between(0.10, 0.90)
    )

    comm_share = comm_share[valid_mask]
    non_share = non_share[valid_mask]
    total = total[valid_mask]

    return comm_share, non_share, total
# =================================================
# COMPUTE
# =================================================

bike_comm_s, bike_non_s, bike_total = build_user_shares(df_jobs_bike_fi)
pt_comm_s, pt_non_s, pt_total       = build_user_shares(df_jobs_pt_fi)
car_comm_s, car_non_s, car_total    = build_user_shares(df_jobs_car_fi)

# =================================================
# MEDIANS
# =================================================

order = ["Bike", "Public Transport", "Car"]

data = pd.DataFrame({
    "Mode": order,
    "Commuting_pct": [
        bike_comm_s.median(),
        pt_comm_s.median(),
        car_comm_s.median()
    ],
    "NonWork_pct": [
        bike_non_s.median(),
        pt_non_s.median(),
        car_non_s.median()
    ],
    "Total": [
        bike_total.median(),
        pt_total.median(),
        car_total.median()
    ]
})

# convert to %
data["Commuting_pct"] *= 100
data["NonWork_pct"] *= 100

# =================================================
# PLOT
# =================================================

colors = {
    "commuting": "#1f2a44",
    "non_work": "#4c78a8"
}

fig, ax = plt.subplots(figsize=(11, 4.6))

y = np.arange(len(order))
bar_h = 0.45

ax.barh(y, data["Commuting_pct"], height=bar_h, color=colors["commuting"])
ax.barh(
    y,
    data["NonWork_pct"],
    left=data["Commuting_pct"],
    height=bar_h,
    color=colors["non_work"]
)

# =================================================
# LABELS
# =================================================

for i in range(len(order)):

    c = data["Commuting_pct"].iloc[i]
    n = data["NonWork_pct"].iloc[i]
    t = data["Total"].iloc[i]

    ax.text(c/2, i, f"{c:.1f}%",
            ha="center", va="center",
            color="white", fontweight="bold")

    ax.text(c + n/2, i, f"{n:.1f}%",
            ha="center", va="center",
            color="white", fontweight="bold")

    ax.text(102, i, f"Total: {t:.2f} kg CO₂",
            va="center", fontsize=10, fontweight="bold")

# =================================================
# FORMATTING
# =================================================

ax.set_yticks(y)
ax.set_yticklabels(order)

ax.set_xlim(0, 100)

ax.set_xlabel("Share of CO₂ footprint per trip purpose (%)")

ax.set_title(
    "Decent mobility CO₂ composition (Finland)\nMedian of per-user shares",
    pad=14
)

ax.xaxis.grid(True, linestyle="--", alpha=0.25)
ax.yaxis.grid(False)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

ax.legend(
    ["Commuting (jobs)", "Non-work trips"],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=2,
    frameon=False
)

plt.tight_layout()

plt.savefig(
    "finland_decent_mobility_corrected_shares_v2.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Typical

In [ ]:
final_df_car_typ = pd.read_parquet("./output/car_expenditure_weekly_typical.parquet")
final_df_pt_typ  = pd.read_parquet("./output/PT_expenditure_weekly_typical.parquet")
final_df_bike_typ = pd.read_parquet("./output/bike_expenditure_weekly_typical.parquet")


In [ ]:
final_df_bike_typ.columns

In [ ]:
final_df_pt_typ.columns

In [ ]:
final_df_car_typ.columns

In [ ]:
final_df_pt_typ

In [ ]:
users = pd.read_parquet("./data/user_pois_pt_1.parquet")

user_home_hex = (
    users[users["is_home"] == 1]
    .drop_duplicates(subset="user_id")
)

In [ ]:
car_typ_hex = final_df_car_typ.merge(
    user_home_hex,
    on="user_id",
    how="left"
)

pt_typ_hex = final_df_pt_typ.merge(
    user_home_hex,
    on="user_id",
    how="left"
)

bike_typ_hex = final_df_bike_typ.merge(
    user_home_hex,
    on="user_id",
    how="left"
)

In [ ]:
pt_hex = (
    pt_typ_hex
    .groupby(["home_gid9", "Nimi", "Posnro"])["total_weekly_co2"]
    .agg(["mean", "count"])
    .reset_index()
)

pt_hex = pt_hex[pt_hex["count"] >= 3]
pt_hex["avg_co2_kg"] = pt_hex["mean"] / 1000

In [ ]:
car_hex = (
    car_typ_hex
    .groupby(["home_gid9", "Nimi", "Posnro"])["total_weekly_co2"]
    .agg(["mean", "count"])
    .reset_index()
)

car_hex = car_hex[car_hex["count"] >= 3]
car_hex["avg_co2_kg"] = car_hex["mean"] / 1000

In [ ]:
bike_hex = (
    bike_typ_hex
    .groupby(["home_gid9", "Nimi", "Posnro"])["total_weekly_co2"]
    .agg(["mean", "count"])
    .reset_index()
)

bike_hex = bike_hex[bike_hex["count"] >= 3]
bike_hex["avg_co2_kg"] = bike_hex["mean"] / 1000

In [ ]:
pt_hex.sort_values("avg_co2_kg").head(1200)

In [ ]:
def h3_to_polygon(h):
    # Returns list of (lat, lng) tuples
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)  
    return Polygon(boundary)

In [ ]:
car_hex["geometry"] = car_hex["home_gid9"].apply(h3_to_polygon)
bike_hex["geometry"] = bike_hex["home_gid9"].apply(h3_to_polygon)
pt_hex["geometry"] = pt_hex["home_gid9"].apply(h3_to_polygon)

In [ ]:
car_gdf  = gpd.GeoDataFrame(car_hex,  geometry="geometry", crs="EPSG:4326").to_crs(3857)
bike_gdf = gpd.GeoDataFrame(bike_hex, geometry="geometry", crs="EPSG:4326").to_crs(3857)
pt_gdf   = gpd.GeoDataFrame(pt_hex,   geometry="geometry", crs="EPSG:4326").to_crs(3857)


In [ ]:
bins = [0, 1, 3, 7, 10, 15, 100]
labels = ["0–1", "1–3", "3–7", "7–10", "10–15", "15+"]

for gdf in [car_gdf, bike_gdf, pt_gdf]:
    gdf["co2_cat"] = pd.cut(
        gdf["avg_co2_kg"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )


In [ ]:
colors = {
    "0–1": "#f7fbff",
    "1–3": "#deebf7",
    "3–7": "#c6dbef",
    "7–10": "#9ecae1",
    "10–15": "#6baed6",
    "15+": "#2171b5"
}


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import pandas as pd

# -----------------------------
# Carbon budget thresholds
# -----------------------------
bins = [0, 1, 3, 7, np.inf]
labels = ["0–1", "1–3", "3–7", "7+"]

# darker green = lower emissions
colors_dict = {
    "0–1": "#00441b",   # dark green (lowest)
    "1–3": "#41ab5d",   # medium green
    "3–7": "#c7e9c0",   # light green
    "7+": "#d73027"     # red (above budget)
}

# -----------------------------
# Categorize CO₂
# -----------------------------
def categorize_co2(gdf, col="avg_co2_kg"):
    gdf = gdf.copy()
    gdf["co2_cat"] = pd.cut(
        gdf[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )
    return gdf

gdf_pt   = categorize_co2(pt_gdf)
gdf_car  = categorize_co2(car_gdf)
gdf_bike = categorize_co2(bike_gdf)

# -----------------------------
# Plot maps (vertical layout)
# -----------------------------
fig, axes = plt.subplots(3, 1, figsize=(12, 24))

for ax, gdf, title in zip(
    axes,
    [gdf_pt, gdf_car, gdf_bike],
    ["Public Transport", "Car", "Bike"]
):

    for cat in labels:
        subset = gdf[gdf["co2_cat"] == cat]
        if not subset.empty:
            subset.plot(
                color=colors_dict[cat],
                linewidth=0.2,
                edgecolor="black",
                alpha=0.9,
                ax=ax
            )

    ctx.add_basemap(ax, source=basemaps.POSITRON)

    ax.set_axis_off()
    ax.set_title(
        f"Weekly CO₂ per Resident Hexagon ({title})",
        fontsize=14
    )

    # Legend
    handles = [
        plt.Line2D([0], [0], marker='s',
                   color=colors_dict[label],
                   linestyle='',
                   markersize=10)
        for label in labels
    ]

    ax.legend(
        handles,
        labels,
        title="Weekly CO₂ (kg)",
        loc="upper right",
        frameon=True
    )

plt.tight_layout()

# -----------------------------
# Save as single high-res image
# -----------------------------
plt.savefig(
    "typical_case_weekly_co2_maps.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import pandas as pd

# -----------------------------
# Carbon budget thresholds
# -----------------------------
bins = [0, 1, 3, 7, np.inf]
labels = ["0–1", "1–3", "3–7", "7+"]

# darker green = lower emissions
colors_dict = {
    "0–1": "#00441b",
    "1–3": "#41ab5d",
    "3–7": "#c7e9c0",
    "7+": "#d73027"
}

# -----------------------------
# Categorize CO₂
# -----------------------------
def categorize_co2(gdf, col="avg_co2_kg"):
    gdf = gdf.copy()

    gdf["co2_cat"] = pd.cut(
        gdf[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    return gdf

gdf_bike = categorize_co2(bike_gdf)
gdf_pt   = categorize_co2(pt_gdf)
gdf_car  = categorize_co2(car_gdf)

# -----------------------------
# PLOT MAPS (HORIZONTAL)
# -----------------------------
fig, axes = plt.subplots(
    1,
    3,
    figsize=(24, 8)
)

for ax, gdf, title in zip(
    axes,
    [gdf_bike, gdf_pt, gdf_car],
    ["Bike", "Public Transport", "Car"]
):

    for cat in labels:

        subset = gdf[gdf["co2_cat"] == cat]

        if not subset.empty:

            subset.plot(
                color=colors_dict[cat],
                linewidth=0.2,
                edgecolor="black",
                alpha=0.9,
                ax=ax
            )

    ctx.add_basemap(
        ax,
        source=basemaps.POSITRON
    )

    ax.set_axis_off()

    ax.set_title(
        f"Weekly CO₂ per Resident Hexagon ({title})",
        fontsize=14
    )

# -----------------------------
# SHARED LEGEND
# -----------------------------
handles = [
    plt.Line2D(
        [0],
        [0],
        marker='s',
        color=colors_dict[label],
        linestyle='',
        markersize=12
    )
    for label in labels
]

fig.legend(
    handles,
    labels,
    title="Weekly CO₂ (kg)",
    loc="lower center",
    ncol=4,
    frameon=True,
    fontsize=11,
    title_fontsize=12
)

# -----------------------------
# LAYOUT
# -----------------------------
plt.tight_layout(rect=[0, 0.06, 1, 1])

# -----------------------------
# SAVE
# -----------------------------
plt.savefig(
    "typical_case_weekly_co2_maps_horizontal.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde
import seaborn as sns

# -----------------------------
# Prepare data (HELSINKI base)
# -----------------------------
car_vals  = car_gdf["avg_co2_kg"].values
pt_vals   = pt_gdf["avg_co2_kg"].values
bike_vals = bike_gdf["avg_co2_kg"].values

car_w  = car_gdf["count"].values
pt_w   = pt_gdf["count"].values
bike_w = bike_gdf["count"].values

# -----------------------------
# Clip car + align weights
# -----------------------------
car_mask = car_vals <= 35
car_vals_clip = car_vals[car_mask]
car_w_clip = car_w[car_mask]

# -----------------------------
# Weighted KDE helper
# -----------------------------
def weighted_kde(values, weights, grid):
    kde = gaussian_kde(values, weights=weights)
    return kde(grid)

# -----------------------------
# Grid
# -----------------------------
x_grid = np.linspace(0, 35, 500)

bike_kde = weighted_kde(bike_vals, bike_w, x_grid)
pt_kde   = weighted_kde(pt_vals, pt_w, x_grid)
car_kde  = weighted_kde(car_vals_clip, car_w_clip, x_grid)

# -----------------------------
# % under 7 kg (user-weighted)
# -----------------------------
pct_bike = np.average(bike_vals <= 7, weights=bike_w) * 100
pct_pt   = np.average(pt_vals   <= 7, weights=pt_w) * 100
pct_car  = np.average(car_vals   <= 7, weights=car_w) * 100

# -----------------------------
# Style
# -----------------------------
sns.set_style("white")

plt.figure(figsize=(7, 4.5))

colors = {
    "bike": "#2b8cbe",
    "pt":   "#7bccc4",
    "car":  "#de2d26"
}

# -----------------------------
# Plot KDEs
# -----------------------------
plt.plot(x_grid, bike_kde, color=colors["bike"], linewidth=2, label="Bike")
plt.fill_between(x_grid, bike_kde, color=colors["bike"], alpha=0.25)

plt.plot(x_grid, pt_kde, color=colors["pt"], linewidth=2, label="Public Transport")
plt.fill_between(x_grid, pt_kde, color=colors["pt"], alpha=0.25)

plt.plot(x_grid, car_kde, color=colors["car"], linewidth=2, label="Car")
plt.fill_between(x_grid, car_kde, color=colors["car"], alpha=0.25)

# -----------------------------
# Carbon threshold
# -----------------------------
plt.axvline(7, color="black", linestyle="--", linewidth=1)

ymax = plt.ylim()[1]
plt.text(7, ymax * 0.92, "7 kg", ha='right', va='top', fontsize=9)

# -----------------------------
# Annotations
# -----------------------------
x_text = 22
y_start = ymax * 0.85
y_step = ymax * 0.08

plt.text(x_text, y_start, f"Bike: {pct_bike:.0f}%", color=colors["bike"], fontsize=10)
plt.text(x_text, y_start - y_step, f"PT: {pct_pt:.0f}%", color=colors["pt"], fontsize=10)
plt.text(x_text, y_start - 2*y_step, f"Car: {pct_car:.0f}%", color=colors["car"], fontsize=10)

# -----------------------------
# Labels
# -----------------------------
plt.xlim(0, 35)
plt.xlabel("Weekly CO₂ per person (kg)")
plt.ylabel("Density")
plt.title("Mobility CO₂ Distribution – Helsinki")

plt.legend(frameon=False)
sns.despine()
plt.tight_layout()

plt.savefig("helsinki_co2_distribution_clean.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# -----------------------------
# DATA (Helsinki — USER LEVEL)
# -----------------------------
car_vals  = car_typ_hex["total_weekly_co2"].to_numpy(dtype=float) / 1000
pt_vals   = pt_typ_hex["total_weekly_co2"].to_numpy(dtype=float) / 1000
bike_vals = bike_typ_hex["total_weekly_co2"].to_numpy(dtype=float) / 1000

# -----------------------------
# KDE helper (NO weights)
# -----------------------------
def kde(values, grid):
    kde = gaussian_kde(values)
    return kde(grid)

# -----------------------------
# GRIDS
# -----------------------------
x_main = np.linspace(0, 20, 400)
x_full = np.linspace(0, 60, 800)

bike_kde = kde(bike_vals, x_main)
pt_kde   = kde(pt_vals, x_main)
car_kde_main = kde(car_vals, x_main)
car_kde_full = kde(car_vals, x_full)

# -----------------------------
# % under 7 kg (true users)
# -----------------------------
pct_bike = (bike_vals <= 7).mean() * 100
pct_pt   = (pt_vals <= 7).mean() * 100
pct_car  = (car_vals <= 7).mean() * 100

# -----------------------------
# STYLE
# -----------------------------
sns.set_style("white")

fig, ax = plt.subplots(figsize=(8, 5))

colors = {
    "bike": "#2b8cbe",
    "pt":   "#7bccc4",
    "car":  "#de2d26"
}

# -----------------------------
# MAIN KDE
# -----------------------------
ax.plot(x_main, bike_kde, color=colors["bike"], lw=2, label="Bike")
ax.fill_between(x_main, bike_kde, color=colors["bike"], alpha=0.25)

ax.plot(x_main, pt_kde, color=colors["pt"], lw=2, label="PT")
ax.fill_between(x_main, pt_kde, color=colors["pt"], alpha=0.25)

ax.plot(x_main, car_kde_main, color=colors["car"], lw=2, label="Car")
ax.fill_between(x_main, car_kde_main, color=colors["car"], alpha=0.25)

# -----------------------------
# 7 KG LINE
# -----------------------------
ax.axvline(7, color="black", linestyle="--", linewidth=1)

ax.text(
    7,
    ax.get_ylim()[1] * 0.95,
    "7 kg",
    ha="right",
    va="top",
    fontsize=9
)

# -----------------------------
# AXES
# -----------------------------
ax.set_xlim(0, 20)
ax.set_ylim(bottom=0)
ax.set_xlabel("Weekly CO₂ per person (kg)")
ax.set_ylabel("Density")
ax.set_title("Mobility CO₂ Distribution (Helsinki — Users)")

ax.legend(frameon=False, loc="center right", bbox_to_anchor=(1.0, 0.5))

# -----------------------------
# SUMMARY (RIGHT-ALIGNED)
# -----------------------------
x_pos = 0.98

ax.text(x_pos, 0.38, f"Bike: {pct_bike:.0f}% < 7 kg",
        transform=ax.transAxes, ha="right", color=colors["bike"])

ax.text(x_pos, 0.31, f"PT: {pct_pt:.0f}% < 7 kg",
        transform=ax.transAxes, ha="right", color=colors["pt"])

ax.text(x_pos, 0.24, f"Car: {pct_car:.0f}% < 7 kg",
        transform=ax.transAxes, ha="right", color=colors["car"])

# -----------------------------
# INSET (CAR ZOOM-OUT)
# -----------------------------
ax_inset = inset_axes(
    ax,
    width="28%",
    height="28%",
    loc="upper right",
    borderpad=1.6
)

ax_inset.plot(x_full, car_kde_full, color=colors["car"], lw=2)
ax_inset.fill_between(x_full, car_kde_full, color=colors["car"], alpha=0.3)

ax_inset.axvline(7, color="black", linestyle="--", linewidth=1)

ax_inset.set_xlim(0, 60)
ax_inset.set_ylim(bottom=0)
ax_inset.set_title("Car zoom-out", fontsize=9)
ax_inset.set_xticks([0, 20, 40])

sns.despine()
plt.tight_layout()

plt.savefig("helsinki_user_kde_typ_hex.png", dpi=300, bbox_inches="tight")
plt.show()

# Turku

In [ ]:
final_df_car_typ_turku = pd.read_parquet("./output/car_expenditure_weekly_typical_turku.parquet")
final_df_pt_typ_turku  = pd.read_parquet("./output/PT_expenditure_weekly_typical_turku.parquet")
final_df_bike_typ_turku = pd.read_parquet("./output/bike_expenditure_weekly_typical_turku.parquet")

In [ ]:
final_df_car_typ_turku

In [ ]:
users = pd.read_parquet("./data/user_pois_pt_1_turku.parquet")

user_home_hex_turku = (
    users[users["is_home"] == 1]
    .drop_duplicates(subset="user_id")
)

In [ ]:
user_home_hex_turku

In [ ]:
user_home_hex_turku = (
    user_home_hex_turku[["user_id", "home_gid9"]]
    .drop_duplicates()
)

In [ ]:
user_home_hex_turku["geometry"] = user_home_hex_turku["home_gid9"].apply(h3_to_polygon)

user_home_hex_turku = gpd.GeoDataFrame(
    user_home_hex_turku,
    geometry="geometry",
    crs="EPSG:4326"
)

#user_home_hex_turku.explore()

In [ ]:
car_typ_hex_turku = final_df_car_typ_turku.merge(
    user_home_hex_turku,
    on="user_id",
    how="left"
)


In [ ]:

pt_typ_hex_turku = final_df_pt_typ_turku.merge(
    user_home_hex_turku,
    on="user_id",
    how="left"
)

bike_typ_hex_turku = final_df_bike_typ_turku.merge(
    user_home_hex_turku,
    on="user_id",
    how="left"
)

In [ ]:
final_df_pt_typ_turku

In [ ]:
pt_hex_turku = (
    pt_typ_hex_turku
    .groupby(["home_gid9", "nimi", "postinumer"])["total_weekly_co2"]
    .agg(["mean", "count"])
    .reset_index()
)

pt_hex_turku = pt_hex_turku[pt_hex_turku["count"] >= 1]
pt_hex_turku["avg_co2_kg"] = pt_hex_turku["mean"] / 1000

In [ ]:
car_hex_turku = (
    car_typ_hex_turku
    .groupby(["home_gid9", "Nimi", "Posnro"])["total_weekly_co2"]
    .agg(["mean", "count"])
    .reset_index()
)

car_hex_turku = car_hex_turku[car_hex_turku["count"] >= 3]
car_hex_turku["avg_co2_kg"] = car_hex_turku["mean"] / 1000

In [ ]:
bike_hex_turku = (
    bike_typ_hex_turku
    .groupby(["home_gid9", "Nimi", "Posnro"])["total_weekly_co2"]
    .agg(mean_co2="mean", count="count")
    .reset_index()
)

bike_hex_turku = bike_hex_turku[bike_hex_turku["count"] >= 3]
bike_hex_turku["avg_co2_kg"] = bike_hex_turku["mean_co2"] / 1000

In [ ]:
def h3_to_polygon(h):
    # Returns list of (lat, lng) tuples
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)  
    return Polygon(boundary)

In [ ]:
car_hex_turku["geometry"] = car_hex_turku["home_gid9"].apply(h3_to_polygon)
bike_hex_turku["geometry"] = bike_hex_turku["home_gid9"].apply(h3_to_polygon)
pt_hex_turku["geometry"] = pt_hex_turku["home_gid9"].apply(h3_to_polygon)

In [ ]:
car_gdf_turku  = gpd.GeoDataFrame(car_hex_turku,  geometry="geometry", crs="EPSG:4326").to_crs(3857)
bike_gdf_turku = gpd.GeoDataFrame(bike_hex_turku, geometry="geometry", crs="EPSG:4326").to_crs(3857)
pt_gdf_turku   = gpd.GeoDataFrame(pt_hex_turku,   geometry="geometry", crs="EPSG:4326").to_crs(3857)

In [ ]:
bins = [0, 1, 3, 7, 10, 15, 100]
labels = ["0–1", "1–3", "3–7", "7–10", "10–15", "15+"]

for gdf in [car_gdf_turku, bike_gdf_turku, pt_gdf_turku]:
    gdf["co2_cat"] = pd.cut(
        gdf["avg_co2_kg"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

In [ ]:
colors = {
    "0–1": "#f7fbff",
    "1–3": "#deebf7",
    "3–7": "#c6dbef",
    "7–10": "#9ecae1",
    "10–15": "#6baed6",
    "15+": "#2171b5"
}


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import pandas as pd
import geopandas as gpd

# -------------------------------------------------
# ENSURE SAME CRS
# -------------------------------------------------
pt_gdf_turku = pt_gdf_turku.to_crs(epsg=3857)
car_gdf_turku = car_gdf_turku.to_crs(epsg=3857)
bike_gdf_turku = bike_gdf_turku.to_crs(epsg=3857)

# -------------------------------------------------
# LOAD FILTER POLYGON
# -------------------------------------------------
filter_poly = gpd.read_parquet(
    "./data/filter_hex_turku.parquet"
).to_crs(epsg=3857)

# NEW geopandas syntax
selection_geom = filter_poly.union_all()

In [ ]:
filter_poly

In [ ]:


# -------------------------------------------------
# FILTER ONLY DISPLAYED HEXAGONS
# -------------------------------------------------
gdf_pt_turku = pt_gdf_turku[
    pt_gdf_turku.geometry.centroid.within(selection_geom)
].copy()

gdf_car_turku = car_gdf_turku[
    car_gdf_turku.geometry.centroid.within(selection_geom)
].copy()

gdf_bike_turku = bike_gdf_turku[
    bike_gdf_turku.geometry.centroid.within(selection_geom)
].copy()

# -------------------------------------------------
# Carbon budget thresholds
# -------------------------------------------------
bins = [0, 1, 3, 7, np.inf]

labels = [
    "0–1",
    "1–3",
    "3–7",
    "7+"
]

colors_dict = {
    "0–1": "#00441b",
    "1–3": "#41ab5d",
    "3–7": "#c7e9c0",
    "7+": "#d73027"
}

# -------------------------------------------------
# Categorize CO₂
# -------------------------------------------------
def categorize_co2(gdf, col="avg_co2_kg"):

    gdf = gdf.copy()

    gdf["co2_cat"] = pd.cut(
        gdf[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    return gdf

gdf_pt_turku = categorize_co2(gdf_pt_turku)
gdf_car_turku = categorize_co2(gdf_car_turku)
gdf_bike_turku = categorize_co2(gdf_bike_turku)

# -------------------------------------------------
# PLOT MAPS
# -------------------------------------------------
fig, axes = plt.subplots(3, 1, figsize=(12, 24))

for ax, gdf, title in zip(

    axes,

    [
        gdf_pt_turku,
        gdf_car_turku,
        gdf_bike_turku
    ],

    [
        "Public Transport",
        "Car",
        "Bike"
    ]
):

    # ---------------------------------------------
    # plot categories
    # ---------------------------------------------
    for cat in labels:

        subset = gdf[gdf["co2_cat"] == cat]

        if not subset.empty:

            subset.plot(
                color=colors_dict[cat],
                linewidth=0.2,
                edgecolor="black",
                alpha=0.9,
                ax=ax
            )

    # ---------------------------------------------
    # basemap
    # IMPORTANT: fixed zoom
    # ---------------------------------------------
    ctx.add_basemap(
        ax,
        source=basemaps.POSITRON,
        zoom=11
    )

    # ---------------------------------------------
    # aesthetics
    # ---------------------------------------------
    ax.set_axis_off()

    ax.set_title(
        f"Weekly CO₂ per Resident Hexagon ({title})",
        fontsize=14
    )

    # ---------------------------------------------
    # legend
    # ---------------------------------------------
    handles = [

        plt.Line2D(
            [0],
            [0],
            marker='s',
            color=colors_dict[label],
            linestyle='',
            markersize=10
        )

        for label in labels
    ]

    ax.legend(
        handles,
        labels,
        title="Weekly CO₂ (kg)",
        loc="upper right",
        frameon=True
    )

# -------------------------------------------------
# FINALIZE
# -------------------------------------------------
plt.tight_layout()

plt.savefig(
    "typical_case_weekly_co2_maps_turku_filtered.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
# -------------------------------------------------
# FILTER ONLY DISPLAYED HEXAGONS
# -------------------------------------------------
gdf_pt_turku = pt_gdf_turku[
    pt_gdf_turku.geometry.centroid.within(selection_geom)
].copy()

gdf_car_turku = car_gdf_turku[
    car_gdf_turku.geometry.centroid.within(selection_geom)
].copy()

gdf_bike_turku = bike_gdf_turku[
    bike_gdf_turku.geometry.centroid.within(selection_geom)
].copy()

# -------------------------------------------------
# Carbon budget thresholds
# -------------------------------------------------
bins = [0, 1, 3, 7, np.inf]

labels = [
    "0–1",
    "1–3",
    "3–7",
    "7+"
]

colors_dict = {
    "0–1": "#00441b",
    "1–3": "#41ab5d",
    "3–7": "#c7e9c0",
    "7+": "#d73027"
}

# -------------------------------------------------
# Categorize CO₂
# -------------------------------------------------
def categorize_co2(gdf, col="avg_co2_kg"):

    gdf = gdf.copy()

    gdf["co2_cat"] = pd.cut(
        gdf[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    return gdf

gdf_bike_turku = categorize_co2(gdf_bike_turku)
gdf_pt_turku = categorize_co2(gdf_pt_turku)
gdf_car_turku = categorize_co2(gdf_car_turku)

# -------------------------------------------------
# PLOT MAPS (HORIZONTAL)
# -------------------------------------------------
fig, axes = plt.subplots(
    1,
    3,
    figsize=(24, 8)
)

for ax, gdf, title in zip(

    axes,

    [
        gdf_bike_turku,
        gdf_pt_turku,
        gdf_car_turku
    ],

    [
        "Bike",
        "Public Transport",
        "Car"
    ]
):

    # ---------------------------------------------
    # plot categories
    # ---------------------------------------------
    for cat in labels:

        subset = gdf[gdf["co2_cat"] == cat]

        if not subset.empty:

            subset.plot(
                color=colors_dict[cat],
                linewidth=0.2,
                edgecolor="black",
                alpha=0.9,
                ax=ax
            )

    # ---------------------------------------------
    # basemap
    # ---------------------------------------------
    ctx.add_basemap(
        ax,
        source=basemaps.POSITRON,
        zoom=11
    )

    # ---------------------------------------------
    # aesthetics
    # ---------------------------------------------
    ax.set_axis_off()

    ax.set_title(
        f"Weekly CO₂ per Resident Hexagon ({title})",
        fontsize=14
    )

# -------------------------------------------------
# SHARED LEGEND
# -------------------------------------------------
handles = [

    plt.Line2D(
        [0],
        [0],
        marker='s',
        color=colors_dict[label],
        linestyle='',
        markersize=12
    )

    for label in labels
]

fig.legend(
    handles,
    labels,
    title="Weekly CO₂ (kg)",
    loc="lower center",
    ncol=4,
    frameon=True,
    fontsize=11,
    title_fontsize=12
)

# -------------------------------------------------
# FINALIZE
# -------------------------------------------------
plt.tight_layout(rect=[0, 0.06, 1, 1])

plt.savefig(
    "typical_case_weekly_co2_maps_turku_horizontal.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import pandas as pd

# -----------------------------
# Carbon budget thresholds
# -----------------------------
bins = [0, 1, 3, 7, np.inf]
labels = ["0–1", "1–3", "3–7", "7+"]

colors_dict = {
    "0–1": "#00441b",
    "1–3": "#41ab5d",
    "3–7": "#c7e9c0",
    "7+": "#d73027"
}

# -----------------------------
# Categorize CO₂
# -----------------------------
def categorize_co2(gdf, col="avg_co2_kg"):
    gdf = gdf.copy()
    gdf["co2_cat"] = pd.cut(
        gdf[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )
    return gdf

gdf_pt_turku   = categorize_co2(pt_gdf_turku)
gdf_car_turku  = categorize_co2(car_gdf_turku)
gdf_bike_turku = categorize_co2(bike_gdf_turku)

# -----------------------------
# Plot maps (vertical layout)
# -----------------------------
fig, axes = plt.subplots(3, 1, figsize=(12, 24))

for ax, gdf, title in zip(
    axes,
    [gdf_pt_turku, gdf_car_turku, gdf_bike_turku],
    ["Public Transport", "Car", "Bike"]
):

    for cat in labels:
        subset = gdf[gdf["co2_cat"] == cat]
        if not subset.empty:
            subset.plot(
                color=colors_dict[cat],
                linewidth=0.2,
                edgecolor="black",
                alpha=0.9,
                ax=ax
            )

    ctx.add_basemap(ax, source=basemaps.POSITRON)

    ax.set_axis_off()
    ax.set_title(
        f"Weekly CO₂ per Resident Hexagon ({title})",
        fontsize=14
    )

    handles = [
        plt.Line2D([0], [0], marker='s',
                   color=colors_dict[label],
                   linestyle='',
                   markersize=10)
        for label in labels
    ]

    ax.legend(
        handles,
        labels,
        title="Weekly CO₂ (kg)",
        loc="upper right",
        frameon=True
    )

plt.tight_layout()

plt.savefig(
    "typical_case_weekly_co2_maps_turku.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# -----------------------------
# DATA (Turku — USER LEVEL)
# -----------------------------
car_vals  = car_typ_hex_turku["total_weekly_co2"].to_numpy(dtype=float) / 1000
pt_vals   = pt_typ_hex_turku["total_weekly_co2"].to_numpy(dtype=float) / 1000
bike_vals = bike_typ_hex_turku["total_weekly_co2"].to_numpy(dtype=float) / 1000

# -----------------------------
# KDE helper
# -----------------------------
def kde(values, grid):
    kde = gaussian_kde(values)
    return kde(grid)

# -----------------------------
# GRIDS
# -----------------------------
x_main = np.linspace(0, 20, 400)
x_full = np.linspace(0, 60, 800)

bike_kde = kde(bike_vals, x_main)
pt_kde   = kde(pt_vals, x_main)
car_kde_main = kde(car_vals, x_main)
car_kde_full = kde(car_vals, x_full)

# -----------------------------
# % under 7 kg
# -----------------------------
pct_bike = (bike_vals <= 7).mean() * 100
pct_pt   = (pt_vals <= 7).mean() * 100
pct_car  = (car_vals <= 7).mean() * 100

# -----------------------------
# STYLE
# -----------------------------
sns.set_style("white")

fig, ax = plt.subplots(figsize=(8, 5))

colors = {
    "bike": "#2b8cbe",
    "pt":   "#7bccc4",
    "car":  "#de2d26"
}

# -----------------------------
# MAIN KDE
# -----------------------------
ax.plot(x_main, bike_kde, color=colors["bike"], lw=2, label="Bike")
ax.fill_between(x_main, bike_kde, color=colors["bike"], alpha=0.25)

ax.plot(x_main, pt_kde, color=colors["pt"], lw=2, label="PT")
ax.fill_between(x_main, pt_kde, color=colors["pt"], alpha=0.25)

ax.plot(x_main, car_kde_main, color=colors["car"], lw=2, label="Car")
ax.fill_between(x_main, car_kde_main, color=colors["car"], alpha=0.25)

# -----------------------------
# 7 KG LINE
# -----------------------------
ax.axvline(7, color="black", linestyle="--", linewidth=1)

ax.text(
    7,
    ax.get_ylim()[1] * 0.95,
    "7 kg",
    ha="right",
    va="top",
    fontsize=9
)

# -----------------------------
# AXES
# -----------------------------
ax.set_xlim(0, 20)
ax.set_ylim(bottom=0)
ax.set_xlabel("Weekly CO₂ per person (kg)")
ax.set_ylabel("Density")
ax.set_title("Mobility CO₂ Distribution (Turku — Users)")

ax.legend(frameon=False, loc="center right", bbox_to_anchor=(1.0, 0.5))

# -----------------------------
# SUMMARY
# -----------------------------
x_pos = 0.98

ax.text(x_pos, 0.38, f"Bike: {pct_bike:.0f}% < 7 kg",
        transform=ax.transAxes, ha="right", color=colors["bike"])

ax.text(x_pos, 0.31, f"PT: {pct_pt:.0f}% < 7 kg",
        transform=ax.transAxes, ha="right", color=colors["pt"])

ax.text(x_pos, 0.24, f"Car: {pct_car:.0f}% < 7 kg",
        transform=ax.transAxes, ha="right", color=colors["car"])

# -----------------------------
# INSET (CAR ZOOM-OUT)
# -----------------------------
ax_inset = inset_axes(
    ax,
    width="28%",
    height="28%",
    loc="upper right",
    borderpad=1.6
)

ax_inset.plot(x_full, car_kde_full, color=colors["car"], lw=2)
ax_inset.fill_between(x_full, car_kde_full, color=colors["car"], alpha=0.3)

ax_inset.axvline(7, color="black", linestyle="--", linewidth=1)

ax_inset.set_xlim(0, 60)
ax_inset.set_ylim(bottom=0)
ax_inset.set_title("Car zoom-out", fontsize=9)
ax_inset.set_xticks([0, 20, 40])

sns.despine()
plt.tight_layout()

plt.savefig("turku_user_kde_typ_hex.png", dpi=300, bbox_inches="tight")
plt.show()

# Tampere

In [ ]:
final_df_car_typ_tampere = pd.read_parquet("./output/car_expenditure_weekly_typical_tampere.parquet")
final_df_pt_typ_tampere  = pd.read_parquet("./output/PT_expenditure_weekly_typical_tampere.parquet")
final_df_bike_typ_tampere = pd.read_parquet("./output/bike_expenditure_weekly_typical_tampere.parquet")

In [ ]:
users = pd.read_parquet("./data/user_pois_pt_1_tampere.parquet")

user_home_hex_tampere = (
    users[users["is_home"] == 1]
    .drop_duplicates(subset="user_id")
)

In [ ]:
user_home_hex_tampere = (
    user_home_hex_tampere[["user_id", "home_gid9"]]
    .drop_duplicates()
)

In [ ]:
user_home_hex_tampere

In [ ]:
user_home_hex_tampere["geometry"] = user_home_hex_tampere["home_gid9"].apply(h3_to_polygon)

user_home_hex_tampere = gpd.GeoDataFrame(
    user_home_hex_tampere,
    geometry="geometry",
    crs="EPSG:4326"
)

user_home_hex_tampere.explore()

In [ ]:
car_typ_hex_tampere = final_df_car_typ_tampere.merge(
    user_home_hex_tampere,
    on="user_id",
    how="left"
)

pt_typ_hex_tampere = final_df_pt_typ_tampere.merge(
    user_home_hex_tampere,
    on="user_id",
    how="left"
)

bike_typ_hex_tampere = final_df_bike_typ_tampere.merge(
    user_home_hex_tampere,
    on="user_id",
    how="left"
)

In [ ]:
def h3_to_polygon(h):
    # Returns list of (lat, lng) tuples
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)  
    return Polygon(boundary)

In [ ]:
pt_hex_tampere = (
    pt_typ_hex_tampere
    .groupby(["home_gid9", "nimi", "postinumer"])["total_weekly_co2"]
    .agg(["mean", "count"])
    .reset_index()
)

pt_hex_tampere = pt_hex_tampere[pt_hex_tampere["count"] >= 1]
pt_hex_tampere["avg_co2_kg"] = pt_hex_tampere["mean"] / 1000


car_hex_tampere = (
    car_typ_hex_tampere
    .groupby(["home_gid9", "Nimi", "Posnro"])["total_weekly_co2"]
    .agg(["mean", "count"])
    .reset_index()
)

car_hex_tampere = car_hex_tampere[car_hex_tampere["count"] >= 1]
car_hex_tampere["avg_co2_kg"] = car_hex_tampere["mean"] / 1000


bike_hex_tampere = (
    bike_typ_hex_tampere
    .groupby(["home_gid9", "Nimi", "Posnro"])["total_weekly_co2"]
    .agg(mean_co2="mean", count="count")
    .reset_index()
)

bike_hex_tampere = bike_hex_tampere[bike_hex_tampere["count"] >= 1]
bike_hex_tampere["avg_co2_kg"] = bike_hex_tampere["mean_co2"] / 1000

In [ ]:
car_hex_tampere["geometry"] = car_hex_tampere["home_gid9"].apply(h3_to_polygon)
bike_hex_tampere["geometry"] = bike_hex_tampere["home_gid9"].apply(h3_to_polygon)
pt_hex_tampere["geometry"] = pt_hex_tampere["home_gid9"].apply(h3_to_polygon)

In [ ]:
car_gdf_tampere  = gpd.GeoDataFrame(car_hex_tampere,  geometry="geometry", crs="EPSG:4326").to_crs(3857)
bike_gdf_tampere = gpd.GeoDataFrame(bike_hex_tampere, geometry="geometry", crs="EPSG:4326").to_crs(3857)
pt_gdf_tampere   = gpd.GeoDataFrame(pt_hex_tampere,   geometry="geometry", crs="EPSG:4326").to_crs(3857)

In [ ]:
bins = [0, 1, 3, 7, 10, 15, 100]
labels = ["0–1", "1–3", "3–7", "7–10", "10–15", "15+"]

for gdf in [car_gdf_tampere, bike_gdf_tampere, pt_gdf_tampere]:
    gdf["co2_cat"] = pd.cut(
        gdf["avg_co2_kg"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

In [ ]:
colors = {
    "0–1": "#f7fbff",
    "1–3": "#deebf7",
    "3–7": "#c6dbef",
    "7–10": "#9ecae1",
    "10–15": "#6baed6",
    "15+": "#2171b5"
}


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import pandas as pd

# -----------------------------
# Carbon budget thresholds
# -----------------------------
bins = [0, 1, 3, 7, np.inf]
labels = ["0–1", "1–3", "3–7", "7+"]

colors_dict = {
    "0–1": "#00441b",
    "1–3": "#41ab5d",
    "3–7": "#c7e9c0",
    "7+": "#d73027"
}

# -----------------------------
# Categorize CO₂
# -----------------------------
def categorize_co2(gdf, col="avg_co2_kg"):
    gdf = gdf.copy()
    gdf["co2_cat"] = pd.cut(
        gdf[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )
    return gdf

gdf_pt_tampere   = categorize_co2(pt_gdf_tampere)
gdf_car_tampere  = categorize_co2(car_gdf_tampere)
gdf_bike_tampere = categorize_co2(bike_gdf_tampere)

# -----------------------------
# Plot maps (vertical layout)
# -----------------------------
fig, axes = plt.subplots(3, 1, figsize=(12, 24))

for ax, gdf, title in zip(
    axes,
    [gdf_pt_tampere, gdf_car_tampere, gdf_bike_tampere],
    ["Public Transport", "Car", "Bike"]
):

    for cat in labels:
        subset = gdf[gdf["co2_cat"] == cat]
        if not subset.empty:
            subset.plot(
                color=colors_dict[cat],
                linewidth=0.2,
                edgecolor="black",
                alpha=0.9,
                ax=ax
            )

    ctx.add_basemap(ax, source=basemaps.POSITRON)

    ax.set_axis_off()
    ax.set_title(
        f"Weekly CO₂ per Resident Hexagon ({title})",
        fontsize=14
    )

    handles = [
        plt.Line2D([0], [0], marker='s',
                   color=colors_dict[label],
                   linestyle='',
                   markersize=10)
        for label in labels
    ]

    ax.legend(
        handles,
        labels,
        title="Weekly CO₂ (kg)",
        loc="upper right",
        frameon=True
    )

plt.tight_layout()

plt.savefig(
    "typical_case_weekly_co2_maps_tampere.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import pandas as pd
import geopandas as gpd

# -------------------------------------------------
# ENSURE SAME CRS
# -------------------------------------------------
pt_gdf_tampere = pt_gdf_tampere.to_crs(epsg=3857)
car_gdf_tampere = car_gdf_tampere.to_crs(epsg=3857)
bike_gdf_tampere = bike_gdf_tampere.to_crs(epsg=3857)

# -------------------------------------------------
# LOAD FILTER POLYGON
# -------------------------------------------------
filter_poly = gpd.read_parquet(
    "./data/filter_hex_tampere.parquet"
).to_crs(epsg=3857)

selection_geom = filter_poly.union_all()

# -------------------------------------------------
# FILTER ONLY DISPLAYED HEXAGONS
# -------------------------------------------------
gdf_pt_tampere = pt_gdf_tampere[
    pt_gdf_tampere.geometry.centroid.within(selection_geom)
].copy()

gdf_car_tampere = car_gdf_tampere[
    car_gdf_tampere.geometry.centroid.within(selection_geom)
].copy()

gdf_bike_tampere = bike_gdf_tampere[
    bike_gdf_tampere.geometry.centroid.within(selection_geom)
].copy()

# -------------------------------------------------
# Carbon budget thresholds
# -------------------------------------------------
bins = [0, 1, 3, 7, np.inf]

labels = [
    "0–1",
    "1–3",
    "3–7",
    "7+"
]

colors_dict = {
    "0–1": "#00441b",
    "1–3": "#41ab5d",
    "3–7": "#c7e9c0",
    "7+": "#d73027"
}

# -------------------------------------------------
# Categorize CO₂
# -------------------------------------------------
def categorize_co2(gdf, col="avg_co2_kg"):

    gdf = gdf.copy()

    gdf["co2_cat"] = pd.cut(
        gdf[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    return gdf

gdf_pt_tampere = categorize_co2(gdf_pt_tampere)
gdf_car_tampere = categorize_co2(gdf_car_tampere)
gdf_bike_tampere = categorize_co2(gdf_bike_tampere)

# -------------------------------------------------
# PLOT MAPS
# -------------------------------------------------
fig, axes = plt.subplots(3, 1, figsize=(12, 24))

for ax, gdf, title in zip(

    axes,

    [
        gdf_pt_tampere,
        gdf_car_tampere,
        gdf_bike_tampere
    ],

    [
        "Public Transport",
        "Car",
        "Bike"
    ]
):

    # ---------------------------------------------
    # plot categories
    # ---------------------------------------------
    for cat in labels:

        subset = gdf[gdf["co2_cat"] == cat]

        if not subset.empty:

            subset.plot(
                color=colors_dict[cat],
                linewidth=0.2,
                edgecolor="black",
                alpha=0.9,
                ax=ax
            )

    # ---------------------------------------------
    # basemap
    # ---------------------------------------------
    ctx.add_basemap(
        ax,
        source=basemaps.POSITRON,
        zoom=11
    )

    # ---------------------------------------------
    # aesthetics
    # ---------------------------------------------
    ax.set_axis_off()

    ax.set_title(
        f"Weekly CO₂ per Resident Hexagon ({title})",
        fontsize=14
    )

    # ---------------------------------------------
    # legend
    # ---------------------------------------------
    handles = [

        plt.Line2D(
            [0],
            [0],
            marker='s',
            color=colors_dict[label],
            linestyle='',
            markersize=10
        )

        for label in labels
    ]

    ax.legend(
        handles,
        labels,
        title="Weekly CO₂ (kg)",
        loc="upper right",
        frameon=True
    )

# -------------------------------------------------
# FINALIZE
# -------------------------------------------------
plt.tight_layout()

plt.savefig(
    "typical_case_weekly_co2_maps_tampere_filtered.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import pandas as pd
import geopandas as gpd

# -------------------------------------------------
# ENSURE SAME CRS
# -------------------------------------------------
pt_gdf_tampere = pt_gdf_tampere.to_crs(epsg=3857)
car_gdf_tampere = car_gdf_tampere.to_crs(epsg=3857)
bike_gdf_tampere = bike_gdf_tampere.to_crs(epsg=3857)

# -------------------------------------------------
# LOAD FILTER POLYGON
# -------------------------------------------------
filter_poly = gpd.read_parquet(
    "./data/filter_hex_tampere.parquet"
).to_crs(epsg=3857)

selection_geom = filter_poly.union_all()

# -------------------------------------------------
# FILTER ONLY DISPLAYED HEXAGONS
# -------------------------------------------------
gdf_pt_tampere = pt_gdf_tampere[
    pt_gdf_tampere.geometry.centroid.within(selection_geom)
].copy()

gdf_car_tampere = car_gdf_tampere[
    car_gdf_tampere.geometry.centroid.within(selection_geom)
].copy()

gdf_bike_tampere = bike_gdf_tampere[
    bike_gdf_tampere.geometry.centroid.within(selection_geom)
].copy()

# -------------------------------------------------
# Carbon budget thresholds
# -------------------------------------------------
bins = [0, 1, 3, 7, np.inf]

labels = [
    "0–1",
    "1–3",
    "3–7",
    "7+"
]

colors_dict = {
    "0–1": "#00441b",
    "1–3": "#41ab5d",
    "3–7": "#c7e9c0",
    "7+": "#d73027"
}

# -------------------------------------------------
# Categorize CO₂
# -------------------------------------------------
def categorize_co2(gdf, col="avg_co2_kg"):

    gdf = gdf.copy()

    gdf["co2_cat"] = pd.cut(
        gdf[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    return gdf

gdf_bike_tampere = categorize_co2(gdf_bike_tampere)
gdf_pt_tampere = categorize_co2(gdf_pt_tampere)
gdf_car_tampere = categorize_co2(gdf_car_tampere)

# -------------------------------------------------
# PLOT MAPS (HORIZONTAL)
# -------------------------------------------------
fig, axes = plt.subplots(
    1,
    3,
    figsize=(24, 8)
)

for ax, gdf, title in zip(

    axes,

    [
        gdf_bike_tampere,
        gdf_pt_tampere,
        gdf_car_tampere
    ],

    [
        "Bike",
        "Public Transport",
        "Car"
    ]
):

    # ---------------------------------------------
    # plot categories
    # ---------------------------------------------
    for cat in labels:

        subset = gdf[gdf["co2_cat"] == cat]

        if not subset.empty:

            subset.plot(
                color=colors_dict[cat],
                linewidth=0.2,
                edgecolor="black",
                alpha=0.9,
                ax=ax
            )

    # ---------------------------------------------
    # basemap
    # ---------------------------------------------
    ctx.add_basemap(
        ax,
        source=basemaps.POSITRON,
        zoom=11
    )

    # ---------------------------------------------
    # aesthetics
    # ---------------------------------------------
    ax.set_axis_off()

    ax.set_title(
        f"Weekly CO₂ per Resident Hexagon ({title})",
        fontsize=14
    )

# -------------------------------------------------
# SHARED LEGEND
# -------------------------------------------------
handles = [

    plt.Line2D(
        [0],
        [0],
        marker='s',
        color=colors_dict[label],
        linestyle='',
        markersize=12
    )

    for label in labels
]

fig.legend(
    handles,
    labels,
    title="Weekly CO₂ (kg)",
    loc="lower center",
    ncol=4,
    frameon=True,
    fontsize=11,
    title_fontsize=12
)

# -------------------------------------------------
# FINALIZE
# -------------------------------------------------
plt.tight_layout(rect=[0, 0.06, 1, 1])

plt.savefig(
    "typical_case_weekly_co2_maps_tampere_horizontal.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# -----------------------------
# DATA (Tampere — USER LEVEL)
# -----------------------------
car_vals  = car_typ_hex_tampere["total_weekly_co2"].to_numpy(dtype=float) / 1000
pt_vals   = pt_typ_hex_tampere["total_weekly_co2"].to_numpy(dtype=float) / 1000
bike_vals = bike_typ_hex_tampere["total_weekly_co2"].to_numpy(dtype=float) / 1000

# -----------------------------
# KDE helper
# -----------------------------
def kde(values, grid):
    kde = gaussian_kde(values)
    return kde(grid)

# -----------------------------
# GRIDS
# -----------------------------
x_main = np.linspace(0, 20, 400)
x_full = np.linspace(0, 70, 800)

bike_kde = kde(bike_vals, x_main)
pt_kde   = kde(pt_vals, x_main)
car_kde_main = kde(car_vals, x_main)
car_kde_full = kde(car_vals, x_full)

# -----------------------------
# % under 7 kg
# -----------------------------
pct_bike = (bike_vals <= 7).mean() * 100
pct_pt   = (pt_vals <= 7).mean() * 100
pct_car  = (car_vals <= 7).mean() * 100

# -----------------------------
# STYLE
# -----------------------------
sns.set_style("white")

fig, ax = plt.subplots(figsize=(8, 5))

colors = {
    "bike": "#2b8cbe",
    "pt":   "#7bccc4",
    "car":  "#de2d26"
}

# -----------------------------
# MAIN KDE
# -----------------------------
ax.plot(x_main, bike_kde, color=colors["bike"], lw=2, label="Bike")
ax.fill_between(x_main, bike_kde, color=colors["bike"], alpha=0.25)

ax.plot(x_main, pt_kde, color=colors["pt"], lw=2, label="PT")
ax.fill_between(x_main, pt_kde, color=colors["pt"], alpha=0.25)

ax.plot(x_main, car_kde_main, color=colors["car"], lw=2, label="Car")
ax.fill_between(x_main, car_kde_main, color=colors["car"], alpha=0.25)

# -----------------------------
# 7 KG LINE
# -----------------------------
ax.axvline(7, color="black", linestyle="--", linewidth=1)

ax.text(
    7,
    ax.get_ylim()[1] * 0.95,
    "7 kg",
    ha="right",
    va="top",
    fontsize=9
)

# -----------------------------
# AXES
# -----------------------------
ax.set_xlim(0, 20)
ax.set_ylim(bottom=0)
ax.set_xlabel("Weekly CO₂ per person (kg)")
ax.set_ylabel("Density")
ax.set_title("Mobility CO₂ Distribution (Tampere — Users)")

ax.legend(frameon=False, loc="center right", bbox_to_anchor=(1.0, 0.5))

# -----------------------------
# SUMMARY
# -----------------------------
x_pos = 0.98

ax.text(x_pos, 0.38, f"Bike: {pct_bike:.0f}% < 7 kg",
        transform=ax.transAxes, ha="right", color=colors["bike"])

ax.text(x_pos, 0.31, f"PT: {pct_pt:.0f}% < 7 kg",
        transform=ax.transAxes, ha="right", color=colors["pt"])

ax.text(x_pos, 0.24, f"Car: {pct_car:.0f}% < 7 kg",
        transform=ax.transAxes, ha="right", color=colors["car"])

# -----------------------------
# INSET (CAR ZOOM-OUT)
# -----------------------------
ax_inset = inset_axes(
    ax,
    width="28%",
    height="28%",
    loc="upper right",
    borderpad=1.6
)

ax_inset.plot(x_full, car_kde_full, color=colors["car"], lw=2)
ax_inset.fill_between(x_full, car_kde_full, color=colors["car"], alpha=0.3)

ax_inset.axvline(7, color="black", linestyle="--", linewidth=1)

ax_inset.set_xlim(0, 70)
ax_inset.set_ylim(bottom=0)
ax_inset.set_title("Car zoom-out", fontsize=9)
ax_inset.set_xticks([0, 20, 40, 60])

sns.despine()
plt.tight_layout()

plt.savefig("tampere_user_kde_typ_hex.png", dpi=300, bbox_inches="tight")
plt.show()

# Oulu

In [ ]:
final_df_car_typ_oulu = pd.read_parquet("./output/car_expenditure_weekly_typical_oulu.parquet")
final_df_pt_typ_oulu  = pd.read_parquet("./output/PT_expenditure_weekly_typical_oulu.parquet")
final_df_bike_typ_oulu = pd.read_parquet("./output/bike_expenditure_weekly_typical_oulu.parquet")

In [ ]:
users = pd.read_parquet("./data/user_pois_pt_1_oulu.parquet")

user_home_hex_oulu = (
    users[users["is_home"] == 1]
    .drop_duplicates(subset="user_id")
)

In [ ]:
user_home_hex_oulu = (
    user_home_hex_oulu[["user_id", "home_gid9"]]
    .drop_duplicates()
)

In [ ]:
def h3_to_polygon(h):
    # Returns list of (lat, lng) tuples
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)  
    return Polygon(boundary)

In [ ]:
user_home_hex_oulu["geometry"] = user_home_hex_oulu["home_gid9"].apply(h3_to_polygon)

user_home_hex_oulu = gpd.GeoDataFrame(
    user_home_hex_oulu,
    geometry="geometry",
    crs="EPSG:4326"
)


In [ ]:
car_typ_hex_oulu = final_df_car_typ_oulu.merge(
    user_home_hex_oulu,
    on="user_id",
    how="left"
)

pt_typ_hex_oulu = final_df_pt_typ_oulu.merge(
    user_home_hex_oulu,
    on="user_id",
    how="left"
)

bike_typ_hex_oulu = final_df_bike_typ_oulu.merge(
    user_home_hex_oulu,
    on="user_id",
    how="left"
)

In [ ]:
pt_hex_oulu = (
    pt_typ_hex_oulu
    .groupby(["home_gid9", "nimi", "postinumer"])["total_weekly_co2"]
    .agg(["mean", "count"])
    .reset_index()
)

pt_hex_oulu = pt_hex_oulu[pt_hex_oulu["count"] >= 1]
pt_hex_oulu["avg_co2_kg"] = pt_hex_oulu["mean"] / 1000


car_hex_oulu = (
    car_typ_hex_oulu
    .groupby(["home_gid9", "Nimi", "Posnro"])["total_weekly_co2"]
    .agg(["mean", "count"])
    .reset_index()
)

car_hex_oulu = car_hex_oulu[car_hex_oulu["count"] >= 1]
car_hex_oulu["avg_co2_kg"] = car_hex_oulu["mean"] / 1000


bike_hex_oulu = (
    bike_typ_hex_oulu
    .groupby(["home_gid9", "Nimi", "Posnro"])["total_weekly_co2"]
    .agg(mean_co2="mean", count="count")
    .reset_index()
)

bike_hex_oulu = bike_hex_oulu[bike_hex_oulu["count"] >= 1]
bike_hex_oulu["avg_co2_kg"] = bike_hex_oulu["mean_co2"] / 1000

In [ ]:
car_hex_oulu["geometry"] = car_hex_oulu["home_gid9"].apply(h3_to_polygon)
bike_hex_oulu["geometry"] = bike_hex_oulu["home_gid9"].apply(h3_to_polygon)
pt_hex_oulu["geometry"] = pt_hex_oulu["home_gid9"].apply(h3_to_polygon)

In [ ]:
car_gdf_oulu  = gpd.GeoDataFrame(car_hex_oulu,  geometry="geometry", crs="EPSG:4326").to_crs(3857)
bike_gdf_oulu = gpd.GeoDataFrame(bike_hex_oulu, geometry="geometry", crs="EPSG:4326").to_crs(3857)
pt_gdf_oulu   = gpd.GeoDataFrame(pt_hex_oulu,   geometry="geometry", crs="EPSG:4326").to_crs(3857)

In [ ]:
bins = [0, 1, 3, 7, 10, 15, 100]
labels = ["0–1", "1–3", "3–7", "7–10", "10–15", "15+"]

for gdf in [car_gdf_oulu, bike_gdf_oulu, pt_gdf_oulu]:
    gdf["co2_cat"] = pd.cut(
        gdf["avg_co2_kg"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

In [ ]:
colors = {
    "0–1": "#f7fbff",
    "1–3": "#deebf7",
    "3–7": "#c6dbef",
    "7–10": "#9ecae1",
    "10–15": "#6baed6",
    "15+": "#2171b5"
}


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import pandas as pd

# -----------------------------
# Carbon budget thresholds
# -----------------------------
bins = [0, 1, 3, 7, np.inf]
labels = ["0–1", "1–3", "3–7", "7+"]

colors_dict = {
    "0–1": "#00441b",
    "1–3": "#41ab5d",
    "3–7": "#c7e9c0",
    "7+": "#d73027"
}

# -----------------------------
# Categorize CO₂
# -----------------------------
def categorize_co2(gdf, col="avg_co2_kg"):
    gdf = gdf.copy()
    gdf["co2_cat"] = pd.cut(
        gdf[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )
    return gdf

gdf_pt_oulu   = categorize_co2(pt_gdf_oulu)
gdf_car_oulu  = categorize_co2(car_gdf_oulu)
gdf_bike_oulu = categorize_co2(bike_gdf_oulu)

# -----------------------------
# Plot maps (vertical layout)
# -----------------------------
fig, axes = plt.subplots(3, 1, figsize=(12, 24))

for ax, gdf, title in zip(
    axes,
    [gdf_pt_oulu, gdf_car_oulu, gdf_bike_oulu],
    ["Public Transport", "Car", "Bike"]
):

    for cat in labels:
        subset = gdf[gdf["co2_cat"] == cat]
        if not subset.empty:
            subset.plot(
                color=colors_dict[cat],
                linewidth=0.2,
                edgecolor="black",
                alpha=0.9,
                ax=ax
            )

    ctx.add_basemap(ax, source=basemaps.POSITRON)

    ax.set_axis_off()
    ax.set_title(
        f"Weekly CO₂ per Resident Hexagon ({title})",
        fontsize=14
    )

    handles = [
        plt.Line2D([0], [0], marker='s',
                   color=colors_dict[label],
                   linestyle='',
                   markersize=10)
        for label in labels
    ]

    ax.legend(
        handles,
        labels,
        title="Weekly CO₂ (kg)",
        loc="upper right",
        frameon=True
    )

plt.tight_layout()

plt.savefig(
    "typical_case_weekly_co2_maps_oulu.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import pandas as pd
import geopandas as gpd

# -------------------------------------------------
# ENSURE SAME CRS
# -------------------------------------------------
pt_gdf_oulu = pt_gdf_oulu.to_crs(epsg=3857)
car_gdf_oulu = car_gdf_oulu.to_crs(epsg=3857)
bike_gdf_oulu = bike_gdf_oulu.to_crs(epsg=3857)

# -------------------------------------------------
# LOAD FILTER POLYGON
# -------------------------------------------------
filter_poly = gpd.read_parquet(
    "./data/filter_hex_oulu.parquet"
).to_crs(epsg=3857)

selection_geom = filter_poly.union_all()

# -------------------------------------------------
# FILTER ONLY DISPLAYED HEXAGONS
# -------------------------------------------------
gdf_pt_oulu = pt_gdf_oulu[
    pt_gdf_oulu.geometry.centroid.within(selection_geom)
].copy()

gdf_car_oulu = car_gdf_oulu[
    car_gdf_oulu.geometry.centroid.within(selection_geom)
].copy()

gdf_bike_oulu = bike_gdf_oulu[
    bike_gdf_oulu.geometry.centroid.within(selection_geom)
].copy()

# -------------------------------------------------
# Carbon budget thresholds
# -------------------------------------------------
bins = [0, 1, 3, 7, np.inf]

labels = [
    "0–1",
    "1–3",
    "3–7",
    "7+"
]

colors_dict = {
    "0–1": "#00441b",
    "1–3": "#41ab5d",
    "3–7": "#c7e9c0",
    "7+": "#d73027"
}

# -------------------------------------------------
# Categorize CO₂
# -------------------------------------------------
def categorize_co2(gdf, col="avg_co2_kg"):

    gdf = gdf.copy()

    gdf["co2_cat"] = pd.cut(
        gdf[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    return gdf

gdf_pt_oulu = categorize_co2(gdf_pt_oulu)
gdf_car_oulu = categorize_co2(gdf_car_oulu)
gdf_bike_oulu = categorize_co2(gdf_bike_oulu)

# -------------------------------------------------
# PLOT MAPS
# -------------------------------------------------
fig, axes = plt.subplots(3, 1, figsize=(12, 24))

for ax, gdf, title in zip(

    axes,

    [
        gdf_pt_oulu,
        gdf_car_oulu,
        gdf_bike_oulu
    ],

    [
        "Public Transport",
        "Car",
        "Bike"
    ]
):

    # ---------------------------------------------
    # plot categories
    # ---------------------------------------------
    for cat in labels:

        subset = gdf[gdf["co2_cat"] == cat]

        if not subset.empty:

            subset.plot(
                color=colors_dict[cat],
                linewidth=0.2,
                edgecolor="black",
                alpha=0.9,
                ax=ax
            )

    # ---------------------------------------------
    # basemap
    # ---------------------------------------------
    ctx.add_basemap(
        ax,
        source=basemaps.POSITRON,
        zoom=11
    )

    # ---------------------------------------------
    # aesthetics
    # ---------------------------------------------
    ax.set_axis_off()

    ax.set_title(
        f"Weekly CO₂ per Resident Hexagon ({title})",
        fontsize=14
    )

    # ---------------------------------------------
    # legend
    # ---------------------------------------------
    handles = [

        plt.Line2D(
            [0],
            [0],
            marker='s',
            color=colors_dict[label],
            linestyle='',
            markersize=10
        )

        for label in labels
    ]

    ax.legend(
        handles,
        labels,
        title="Weekly CO₂ (kg)",
        loc="upper right",
        frameon=True
    )

# -------------------------------------------------
# FINALIZE
# -------------------------------------------------
plt.tight_layout()

plt.savefig(
    "typical_case_weekly_co2_maps_oulu_filtered.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import pandas as pd
import geopandas as gpd

# -------------------------------------------------
# ENSURE SAME CRS
# -------------------------------------------------
pt_gdf_oulu = pt_gdf_oulu.to_crs(epsg=3857)
car_gdf_oulu = car_gdf_oulu.to_crs(epsg=3857)
bike_gdf_oulu = bike_gdf_oulu.to_crs(epsg=3857)

# -------------------------------------------------
# LOAD FILTER POLYGON
# -------------------------------------------------
filter_poly = gpd.read_parquet(
    "./data/filter_hex_oulu.parquet"
).to_crs(epsg=3857)

selection_geom = filter_poly.union_all()

# -------------------------------------------------
# FILTER ONLY DISPLAYED HEXAGONS
# -------------------------------------------------
gdf_pt_oulu = pt_gdf_oulu[
    pt_gdf_oulu.geometry.centroid.within(selection_geom)
].copy()

gdf_car_oulu = car_gdf_oulu[
    car_gdf_oulu.geometry.centroid.within(selection_geom)
].copy()

gdf_bike_oulu = bike_gdf_oulu[
    bike_gdf_oulu.geometry.centroid.within(selection_geom)
].copy()

# -------------------------------------------------
# Carbon budget thresholds
# -------------------------------------------------
bins = [0, 1, 3, 7, np.inf]

labels = [
    "0–1",
    "1–3",
    "3–7",
    "7+"
]

colors_dict = {
    "0–1": "#00441b",
    "1–3": "#41ab5d",
    "3–7": "#c7e9c0",
    "7+": "#d73027"
}

# -------------------------------------------------
# Categorize CO₂
# -------------------------------------------------
def categorize_co2(gdf, col="avg_co2_kg"):

    gdf = gdf.copy()

    gdf["co2_cat"] = pd.cut(
        gdf[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    return gdf

gdf_bike_oulu = categorize_co2(gdf_bike_oulu)
gdf_pt_oulu = categorize_co2(gdf_pt_oulu)
gdf_car_oulu = categorize_co2(gdf_car_oulu)

# -------------------------------------------------
# PLOT MAPS (HORIZONTAL)
# -------------------------------------------------
fig, axes = plt.subplots(
    1,
    3,
    figsize=(24, 8)
)

for ax, gdf, title in zip(

    axes,

    [
        gdf_bike_oulu,
        gdf_pt_oulu,
        gdf_car_oulu
    ],

    [
        "Bike",
        "Public Transport",
        "Car"
    ]
):

    # ---------------------------------------------
    # plot categories
    # ---------------------------------------------
    for cat in labels:

        subset = gdf[gdf["co2_cat"] == cat]

        if not subset.empty:

            subset.plot(
                color=colors_dict[cat],
                linewidth=0.2,
                edgecolor="black",
                alpha=0.9,
                ax=ax
            )

    # ---------------------------------------------
    # basemap
    # ---------------------------------------------
    ctx.add_basemap(
        ax,
        source=basemaps.POSITRON,
        zoom=11
    )

    # ---------------------------------------------
    # aesthetics
    # ---------------------------------------------
    ax.set_axis_off()

    ax.set_title(
        f"Weekly CO₂ per Resident Hexagon ({title})",
        fontsize=14
    )

# -------------------------------------------------
# SHARED LEGEND
# -------------------------------------------------
handles = [

    plt.Line2D(
        [0],
        [0],
        marker='s',
        color=colors_dict[label],
        linestyle='',
        markersize=12
    )

    for label in labels
]

fig.legend(
    handles,
    labels,
    title="Weekly CO₂ (kg)",
    loc="lower center",
    ncol=4,
    frameon=True,
    fontsize=11,
    title_fontsize=12
)

# -------------------------------------------------
# FINALIZE
# -------------------------------------------------
plt.tight_layout(rect=[0, 0.06, 1, 1])

plt.savefig(
    "typical_case_weekly_co2_maps_oulu_horizontal.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
#Summary of images

# Oulu
gdf_pt = categorize_co2(pt_gdf_oulu)
gdf_car = categorize_co2(car_gdf_oulu)
gdf_bike = categorize_co2(bike_gdf_oulu)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

titles = ["Bike", "Public Transport", "Car"]
gdfs = [gdf_bike, gdf_pt, gdf_car]

for ax, gdf, title in zip(axes, gdfs, titles):
    for cat in labels:
        subset = gdf[gdf["co2_cat"] == cat]
        if not subset.empty:
            subset.plot(
                color=colors_dict[cat],
                linewidth=0.1,
                edgecolor="none",
                alpha=0.9,
                ax=ax
            )

    ctx.add_basemap(ax, source=basemaps.POSITRON)
    ax.set_axis_off()
    ax.set_title(title, fontsize=16)

handles = [
    plt.Line2D([0], [0], marker='s',
               color=colors_dict[label],
               linestyle='',
               markersize=10)
    for label in labels
]

fig.legend(
    handles, labels,
    title="Weekly CO₂ (kg)",
    loc="lower center",
    bbox_to_anchor=(0.5, 0.06),
    ncol=4,
    frameon=False
)

plt.tight_layout(rect=[0, 0.06, 1, 1])


plt.show()

In [ ]:
pt_gdf_oulu.sort_values("avg_co2_kg")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# -----------------------------
# DATA (Oulu — USER LEVEL)
# -----------------------------
car_vals  = car_typ_hex_oulu["total_weekly_co2"].to_numpy(dtype=float) / 1000
pt_vals   = pt_typ_hex_oulu["total_weekly_co2"].to_numpy(dtype=float) / 1000
bike_vals = bike_typ_hex_oulu["total_weekly_co2"].to_numpy(dtype=float) / 1000

# -----------------------------
# KDE helper
# -----------------------------
def kde(values, grid):
    kde = gaussian_kde(values)
    return kde(grid)

# -----------------------------
# GRIDS
# -----------------------------
x_main = np.linspace(0, 20, 400)
x_full = np.linspace(0, 70, 800)

bike_kde = kde(bike_vals, x_main)
pt_kde   = kde(pt_vals, x_main)
car_kde_main = kde(car_vals, x_main)
car_kde_full = kde(car_vals, x_full)

# -----------------------------
# % under 7 kg
# -----------------------------
pct_bike = (bike_vals <= 7).mean() * 100
pct_pt   = (pt_vals <= 7).mean() * 100
pct_car  = (car_vals <= 7).mean() * 100

# -----------------------------
# STYLE
# -----------------------------
sns.set_style("white")

fig, ax = plt.subplots(figsize=(8, 5))

colors = {
    "bike": "#2b8cbe",
    "pt":   "#7bccc4",
    "car":  "#de2d26"
}

# -----------------------------
# MAIN KDE
# -----------------------------
ax.plot(x_main, bike_kde, color=colors["bike"], lw=2, label="Bike")
ax.fill_between(x_main, bike_kde, color=colors["bike"], alpha=0.25)

ax.plot(x_main, pt_kde, color=colors["pt"], lw=2, label="PT")
ax.fill_between(x_main, pt_kde, color=colors["pt"], alpha=0.25)

ax.plot(x_main, car_kde_main, color=colors["car"], lw=2, label="Car")
ax.fill_between(x_main, car_kde_main, color=colors["car"], alpha=0.25)

# -----------------------------
# 7 KG LINE
# -----------------------------
ax.axvline(7, color="black", linestyle="--", linewidth=1)

ax.text(
    7,
    ax.get_ylim()[1] * 0.95,
    "7 kg",
    ha="right",
    va="top",
    fontsize=9
)

# -----------------------------
# AXES
# -----------------------------
ax.set_xlim(0, 20)
ax.set_ylim(bottom=0)
ax.set_xlabel("Weekly CO₂ per person (kg)")
ax.set_ylabel("Density")
ax.set_title("Mobility CO₂ Distribution (Oulu — Users)")

ax.legend(frameon=False, loc="center right", bbox_to_anchor=(1.0, 0.5))

# -----------------------------
# SUMMARY
# -----------------------------
x_pos = 0.98

ax.text(x_pos, 0.38, f"Bike: {pct_bike:.0f}% < 7 kg",
        transform=ax.transAxes, ha="right", color=colors["bike"])

ax.text(x_pos, 0.31, f"PT: {pct_pt:.0f}% < 7 kg",
        transform=ax.transAxes, ha="right", color=colors["pt"])

ax.text(x_pos, 0.24, f"Car: {pct_car:.0f}% < 7 kg",
        transform=ax.transAxes, ha="right", color=colors["car"])

# -----------------------------
# INSET (CAR ZOOM-OUT)
# -----------------------------
ax_inset = inset_axes(
    ax,
    width="28%",
    height="28%",
    loc="upper right",
    borderpad=1.6
)

ax_inset.plot(x_full, car_kde_full, color=colors["car"], lw=2)
ax_inset.fill_between(x_full, car_kde_full, color=colors["car"], alpha=0.3)

ax_inset.axvline(7, color="black", linestyle="--", linewidth=1)

ax_inset.set_xlim(0, 70)
ax_inset.set_ylim(bottom=0)
ax_inset.set_title("Car zoom-out", fontsize=9)
ax_inset.set_xticks([0, 20, 40, 60])

sns.despine()
plt.tight_layout()

plt.savefig("oulu_user_kde_typ_hex.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
pt_vals = pt_gdf_oulu["avg_co2_kg"].to_numpy()
pt_w    = pt_gdf_oulu["count"].to_numpy()

In [ ]:
mask = pt_vals <= 7

users_below_7 = pt_w[mask].sum()
total_users = pt_w.sum()

pct_pt = users_below_7 / total_users * 100

In [ ]:
pct_pt

In [ ]:
bike_gap = gdf_bike[['home_gid9','avg_co2_kg','geometry']].merge(
    gdf_bike_nearest[['home_gid9','avg_co2_kg']],
    on='home_gid9',
    suffixes=('_typical','_nearest')
)

bike_gap['gap_bike'] = (
    bike_gap['avg_co2_kg_typical']
    - bike_gap['avg_co2_kg_nearest']
)

In [ ]:
pt_gap = gdf_pt[['home_gid9','avg_co2_kg','geometry']].merge(
    gdf_pt_nearest[['home_gid9','avg_co2_kg']],
    on='home_gid9',
    suffixes=('_typical','_nearest')
)

pt_gap['gap_pt'] = (
    pt_gap['avg_co2_kg_typical']
    - pt_gap['avg_co2_kg_nearest']
)


In [ ]:
car_gap = gdf_car[['home_gid9','avg_co2_kg','geometry']].merge(
    gdf_car_nearest[['home_gid9','avg_co2_kg']],
    on='home_gid9',
    suffixes=('_typical','_nearest')
)

car_gap['gap_car'] = (
    car_gap['avg_co2_kg_typical']
    - car_gap['avg_co2_kg_nearest']
)

In [ ]:
bike_gap['pct_change_bike'] = (
    bike_gap['gap_bike'] / bike_gap['avg_co2_kg_typical']
) * 100

pt_gap['pct_change_pt'] = (
    pt_gap['gap_pt'] / pt_gap['avg_co2_kg_typical']
) * 100

car_gap['pct_change_car'] = (
    car_gap['gap_car'] / car_gap['avg_co2_kg_typical']
) * 100

In [ ]:
import mapclassify
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import matplotlib.patches as mpatches
import numpy as np

# -----------------------------
# Natural breaks for percentage gap
# -----------------------------
n_classes = 5
classifier = mapclassify.NaturalBreaks(car_gap['pct_change_car'], k=n_classes)
car_gap['gap_class'] = classifier.yb  # classes 0..4

# Assign colors per class
colors = plt.cm.RdYlGn_r(np.linspace(0, 1, n_classes))
car_gap['color'] = car_gap['gap_class'].apply(lambda x: colors[x])

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(10,10))

car_gap.plot(
    color=car_gap['color'],
    linewidth=0.2,
    edgecolor='black',
    alpha=0.9,
    ax=ax
)

ctx.add_basemap(ax, source=basemaps.POSITRON)
ax.set_axis_off()
ax.set_title("CO₂ Emissions Gap nearest vs realized — Car (%)", fontsize=14)

# -----------------------------
# Legend with % ranges
# -----------------------------
labels = []
bins = classifier.bins
for i in range(len(bins)):
    if i == 0:
        low = car_gap['pct_change_car'].min()
    else:
        low = bins[i-1]
    high = bins[i]
    labels.append(f"{low:.1f}% – {high:.1f}%")

handles = [mpatches.Patch(color=colors[i], label=labels[i]) for i in range(n_classes)]
ax.legend(handles=handles, title="% Gap per Hexagon", loc="upper right", frameon=True)

plt.show()

In [ ]:
# -----------------------------
# Natural breaks for Bike
# -----------------------------
n_classes = 5
classifier = mapclassify.NaturalBreaks(bike_gap['pct_change_bike'], k=n_classes)
bike_gap['gap_class'] = classifier.yb  # classes 0..4

# Assign colors
colors = plt.cm.RdYlGn_r(np.linspace(0, 1, n_classes))
bike_gap['color'] = bike_gap['gap_class'].apply(lambda x: colors[x])

# Plot
fig, ax = plt.subplots(figsize=(10,10))
bike_gap.plot(
    color=bike_gap['color'],
    linewidth=0.2,
    edgecolor='black',
    alpha=0.9,
    ax=ax
)
ctx.add_basemap(ax, source=basemaps.POSITRON)
ax.set_axis_off()
ax.set_title("CO₂ Emissions Gap nearest vs realized — Bike (%)", fontsize=14)

# Legend with % ranges
labels = []
bins = classifier.bins
for i in range(len(bins)):
    if i == 0:
        low = bike_gap['pct_change_bike'].min()
    else:
        low = bins[i-1]
    high = bins[i]
    labels.append(f"{low:.1f}% – {high:.1f}%")

handles = [mpatches.Patch(color=colors[i], label=labels[i]) for i in range(n_classes)]
ax.legend(handles=handles, title="% Gap per Hexagon", loc="upper right", frameon=True)

plt.show()

In [ ]:
# -----------------------------
# Natural breaks for Public Transport
# -----------------------------
n_classes = 5
classifier = mapclassify.NaturalBreaks(pt_gap['pct_change_pt'], k=n_classes)
pt_gap['gap_class'] = classifier.yb  # classes 0..4

# Assign colors
colors = plt.cm.RdYlGn_r(np.linspace(0, 1, n_classes))
pt_gap['color'] = pt_gap['gap_class'].apply(lambda x: colors[x])

# Plot
fig, ax = plt.subplots(figsize=(10,10))
pt_gap.plot(
    color=pt_gap['color'],
    linewidth=0.2,
    edgecolor='black',
    alpha=0.9,
    ax=ax
)
ctx.add_basemap(ax, source=basemaps.POSITRON)
ax.set_axis_off()
ax.set_title("CO₂ Emissions Gap nearest vs realized — Public Transport (%)", fontsize=14)

# Legend with % ranges
labels = []
bins = classifier.bins
for i in range(len(bins)):
    if i == 0:
        low = pt_gap['pct_change_pt'].min()
    else:
        low = bins[i-1]
    high = bins[i]
    labels.append(f"{low:.1f}% – {high:.1f}%")

handles = [mpatches.Patch(color=colors[i], label=labels[i]) for i in range(n_classes)]
ax.legend(handles=handles, title="% Gap per Hexagon", loc="upper right", frameon=True)

plt.show()

In [ ]:
pt_gap

In [ ]:
pt_gap.sort_values("pct_change_pt")